# 0.0 Introduzione

Lavoro su un dataset di **soddisfazione dei passeggeri di una compagnia aerea**, con l'obiettivo dichiarato di una compagnia attenta sia alla **qualità del servizio** sia alla **riduzione dell'impatto ambientale** del trasporto.

Il task è una **classificazione binaria**: predire se un passeggero sarà `satisfied` oppure `neutral or dissatisfied`, a partire da 22 feature che mescolano caratteristiche del cliente (età, classe di viaggio, tipologia di viaggio, status di fedeltà), del volo (distanza, ritardi) e rating di servizio raccolti via survey (Wi-Fi, sedile, cibo, intrattenimento, ecc.).

Il piano di lavoro segue questa struttura:

1. **EDA**: analisi descrittiva del training set, feature per feature, con commenti interpretativi sulle relazioni con il target;
2. **Preprocessing**: outlier, valori mancanti, gestione dei `0` nei rating, encoding delle categoriche, scaling misto (Robust + Standard);
3. **Feature selection** con tre metodi complementari (T-test, Chi-square, Mutual Information) e definizione di una **selezione ibrida** per la Strada 2;
4. **Modellazione su due strade parallele**: Strada 1 con tutte le feature, Strada 2 con sole 4 feature; spot-check, GridSearch e valutazione sul test set;
5. **Controlli di robustezza**: feature importance del modello vincitore + modello operativo "pre-volo" per stimare la performance senza i rating post-volo (controllo di leakage);
6. **Discussione**: raccomandazioni operative per la compagnia, con un aggancio esplicito al tema sostenibilità.

Il dataset di test viene tenuto da parte per tutta la fase esplorativa e usato solo alla fine per la valutazione dei modelli tunati.

# 1.0 Setup dell'Ambiente

## 1.1 Librerie Utili

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
import warnings
from __future__ import annotations
from typing import List, Optional, Tuple


# Pandas options
pd.options.mode.copy_on_write = True
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# warning
warnings.filterwarnings("ignore")

# riproducibilità di eventuali operazioni random
SEED: int = 0
np.random.seed(SEED)      

## 1.2 Loading dei dati

In [ ]:
raw_data_train = pd.read_csv("train.csv")

**Nota sullo split train/test.** Il dataset, come distribuito dalla sorgente, è già fornito pre-separato in `train.csv` (103.904 record) e `test.csv` (25.976 record), con una proporzione complessiva ~80/20 e la stessa distribuzione della label `satisfaction` in entrambi i file (~43% satisfied). Mantengo questa partizione originaria per riproducibilità e per allinearmi al benchmark pubblico: ogni analisi esplorativa è condotta esclusivamente su `train.csv`, mentre `test.csv` viene tenuto da parte e usato solo nella valutazione finale dei modelli tunati. In questo modo evito ogni forma di information leakage tra le due partizioni.

In [ ]:
raw_data_train.head()

In [ ]:
raw_data_train.info()

In [ ]:
raw_data_train.describe().T

# 2.0 Exploratory Data Analysis

## 2.1 Valori Duplicati

In [ ]:
print("Duplicated rows:", raw_data_train.duplicated().sum())

## 2.2 Valori Mancanti

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Optional

def report_missing_values(
    df: pd.DataFrame,
    *,
    top_n: Optional[int] = 10,
    show_heatmap: bool = True,
    figsize: tuple[int, int] = (12, 6),
) -> pd.Series:
    """
    Analyze and visualize missing values in a DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame to analyze.
    top_n : int | None, default 10
        Number of columns to show in the percentage ranking (None → all).
    show_heatmap : bool, default True
        If True, draws a heatmap of the missing cells.
    figsize : tuple[int, int], default (12, 6)
        Figure size for the heatmap.
    
    Returns
    -------
    pd.Series
        Series with the count of NaNs per column (only those with at least
        one missing value), sorted in descending order.
    """
    # 1. Conteggio dei NaN
    missing_counts = df.isna().sum()
    missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

    print("Missing values per column:")
    print(missing_counts, "\n")

    # 2. Percentuali di NaN
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    if top_n is not None:
        display(missing_pct.head(top_n))
    else:
        display(missing_pct)

    # 3. Heatmap opzionale
    if show_heatmap:
        plt.figure(figsize=figsize)
        sns.heatmap(df.isna(), cbar=False, yticklabels=False)
        plt.title("Map of missing values")
        plt.show()

    return missing_counts

In [ ]:
missing_counts = report_missing_values(raw_data_train, top_n=None, show_heatmap=True)

In [ ]:
raw_data_train = raw_data_train.drop(columns=["Unnamed: 0"])

In [ ]:
raw_data_train.info()

## 2.3 Esploriamo le Features

Per prima cosa, voglio vedere come sono distribuiti i differenti valori tra feature.

In [ ]:
def summarize_feature(
    df: pd.DataFrame,
    column: str,
    /,
    *,
    dropna: bool = False,
    percent_decimals: int = 3
) -> pd.DataFrame:
    """
    Return a summary table (value, count, percent) for a single categorical column.

    Parameters
    ----------
    df : pd.DataFrame
        Source DataFrame.
    column : str
        Name of the column to summarise.
    dropna : bool, default False
        If True, exclude NaN from the counts.
    percent_decimals : int, default 2
        Number of decimal places for the percentage values.

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns:
        - 'value'   : distinct values of the feature (as string)
        - 'count'   : absolute frequency
        - 'percent' : relative frequency in %
        Sorted in descending order by count.
    """
    
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found in DataFrame.")

    total_rows: int = len(df)
    vc = df[column].value_counts(dropna=dropna)

    summary = (
        pd.DataFrame(
            {
                "value": vc.index.astype(str),
                "count": vc.values,
            }
        )
        .assign(percent=lambda d: (d["count"] / total_rows * 100).round(percent_decimals))
        .sort_values(by="count", ascending=False)
        .reset_index(drop=True)
    )

    return summary

In [ ]:
features = raw_data_train.columns

In [ ]:
for feature in features:
    summary = summarize_feature(raw_data_train, feature)
    print(feature)
    print(summary)
    print("-"*30)

## 2.4 Features vs Target

Ora approfondirò l'analisi su ogni singola feature per capire meglio quanto ciascuna di esse possa impattare sulla soddisfazione dell'utente

In [ ]:
# ------------------------ Heuristics ------------------------

def is_rating_0_5(s: pd.Series) -> bool:
    x = pd.to_numeric(s.dropna(), errors="coerce")
    if x.empty:
        return False
    return set(pd.unique(x)).issubset({0, 1, 2, 3, 4, 5})

def is_categorical(s: pd.Series, max_unique: int = 20) -> bool:
    if is_rating_0_5(s):
        return True
    if (pd.api.types.is_object_dtype(s) or
        pd.api.types.is_categorical_dtype(s) or
        pd.api.types.is_bool_dtype(s)):
        return True
    return s.nunique(dropna=True) <= max_unique

def collapse_rare_categories(s: pd.Series, top_n: int = 15) -> pd.Series:
    s = s.astype("object")
    vc = s.value_counts(dropna=False)
    if len(vc) <= top_n:
        return s.fillna("NaN")
    keep = set(vc.index[:top_n])
    out = s.where(s.isin(keep), "Other")
    return out.fillna("NaN")

# ------------------------ Data prep ------------------------

def _count_and_percent(
    x: pd.Series, y: pd.Series, classes: Optional[List] = None
) -> Tuple[pd.Index, List, pd.DataFrame, pd.DataFrame]:
    """
    Restituisce:
      - index categorie X
      - lista classi Y
      - counts_df: conteggi per (x, classe)
      - perc_df: percentuali riga (somma=100 per categoria x)
    """
    ct = pd.crosstab(x, y, dropna=False)
    if classes is not None:
        for cls in classes:
            if cls not in ct.columns:
                ct[cls] = 0
        ct = ct[classes]
    counts_df = ct.copy()
    perc_df = (ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0) * 100).fillna(0.0)
    return counts_df.index, counts_df.columns.tolist(), counts_df, perc_df

# ------------------------ Plotters ------------------------

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Tuple

# --- utilities già definite in precedenza (riutilizzale) ---
# is_rating_0_5, collapse_rare_categories, _count_and_percent

def make_categorical_stacked_figure(
    df: pd.DataFrame,
    feature: str,
    target: str = "satisfaction",
    *,
    show_text: bool = True,       # mostra SOLO la percentuale nel segmento
    bargap: float = 0.35,         # separazione tra barre (0–5 restano “staccate”)
    top_n_cat: int = 15,          # per categoriche non 0..5
    annotate_totals: bool = True  # annota il totale in cima a ciascuna barra
) -> go.Figure:
    """
    Stacked bar: altezza = CONTEGGI reali (differenze tra valori visibili).
    Sulle barre mostriamo SOLO la percentuale della classe dentro il valore.
    Nel tooltip compaiono SIA conteggio assoluto SIA percentuale.
    """
    s_feat = df[feature]
    s_tgt  = df[target]
    classes_sorted = sorted(s_tgt.dropna().unique())

    # --- 0–5: categorie fisse in ordine naturale ---
    if is_rating_0_5(s_feat):
        vals = pd.to_numeric(s_feat, errors="coerce").where(lambda x: x.isin([0,1,2,3,4,5]))
        categories, classes, counts_df, perc_df = _count_and_percent(vals, s_tgt, classes=classes_sorted)
        all_cats = pd.Index([0,1,2,3,4,5], dtype="int64")
        counts_df = counts_df.reindex(index=all_cats, fill_value=0)
        perc_df   = perc_df.reindex(index=all_cats, fill_value=0.0)
        categories = counts_df.index
        x_categoryorder = "array"
        x_categoryarray = [0,1,2,3,4,5]
    else:
        # --- altre categoriche: collassa rare, ordina per volume totale ---
        s_feat = collapse_rare_categories(s_feat, top_n=top_n_cat)
        categories, classes, counts_df, perc_df = _count_and_percent(s_feat, s_tgt, classes=classes_sorted)
        order = counts_df.sum(axis=1).sort_values(ascending=False).index.tolist()
        counts_df = counts_df.loc[order]
        perc_df   = perc_df.loc[order]
        categories = counts_df.index
        x_categoryorder = "category descending"
        x_categoryarray = None

    fig = go.Figure()
    for cls in classes:
        y_counts = counts_df[cls].values         # altezza = conteggi
        y_perc   = perc_df[cls].values           # testo = percentuale
        customdata = np.column_stack([y_counts, y_perc])

        # 👇 SOLO percentuale visibile sulla barra
        texttemplate = "%{customdata[1]:.1f}%" if show_text else None

        fig.add_trace(go.Bar(
            x=categories,
            y=y_counts,
            name=f"Classe {cls}",
            customdata=customdata,
            texttemplate=texttemplate,            # ← niente numero assoluto nel testo
            textposition="inside" if show_text else None,
            hovertemplate=(
                f"{feature}=%{{x}}<br>{target}={cls}"
                "<br>Conteggio=%{customdata[0]}"  # ← il numero assoluto resta nel tooltip
                "<br>Percentuale=%{customdata[1]:.2f}%"
                "<extra></extra>"
            )
        ))

    # Annotazione del totale per ciascuna barra (opzionale, utile per volumi)
    totals = counts_df.sum(axis=1).values
    if annotate_totals and totals.size > 0:
        y_offset = 0.02 * totals.max() if totals.max() > 0 else 1
        for x_val, tot in zip(categories, totals):
            fig.add_annotation(
                x=x_val, y=tot + y_offset, text=str(int(tot)),
                showarrow=False, yanchor="bottom", font=dict(size=11)
            )

    fig.update_layout(
        barmode="stack",
        bargap=bargap,
        template="plotly_white",
        title=f"{feature} per {target} – stacked (conteggi) con % visibili",
        xaxis_title=feature,
        yaxis_title="Conteggio",
        legend_title=target
    )
    if x_categoryarray is not None:
        fig.update_xaxes(categoryorder=x_categoryorder, categoryarray=x_categoryarray)
    else:
        fig.update_xaxes(categoryorder=x_categoryorder)

    return fig


def make_numeric_hist_overlay_figure(
    df: pd.DataFrame,
    feature: str,
    target: str = "satisfaction",
    *,
    nbins: int = 50,
    opacity: float = 0.5,
    density: bool = False
) -> go.Figure:
    s_all = pd.to_numeric(df[feature], errors="coerce").dropna()
    if s_all.empty:
        raise ValueError(f"[{feature}] nessun dato numerico valido.")
    fmin, fmax = float(s_all.min()), float(s_all.max())
    if fmin == fmax:
        fmax = fmin + 1e-9
    bin_edges = np.linspace(fmin, fmax, nbins + 1)
    bin_size  = (fmax - fmin) / nbins

    fig = go.Figure()
    for cls in sorted(df[target].dropna().unique()):
        subset = pd.to_numeric(df.loc[df[target] == cls, feature], errors="coerce").dropna()
        if subset.empty:
            continue
        fig.add_trace(go.Histogram(
            x=subset,
            name=f"Classe {cls}",
            opacity=opacity,
            xbins=dict(start=bin_edges[0], end=bin_edges[-1], size=bin_size),
            histnorm="probability density" if density else None,
            hovertemplate=(
                f"{target}={cls}<br>{feature}=%{{x}}"
                + ("<br>Densità=%{y:.4f}" if density else "<br>Conteggio=%{y}")
                + "<extra></extra>"
            ),
        ))
    fig.update_layout(
        barmode="overlay",
        template="plotly_white",
        title=f"Distribuzione di {feature} per Classe",
        xaxis_title=feature,
        yaxis_title="Densità" if density else "Conteggio",
        legend_title=target
    )
    return fig

# ------------------------ Router (uno alla volta) ------------------------

def make_feature_by_target_figure(
    df: pd.DataFrame,
    feature: str,
    target: str = "satisfaction",
    *,
    cat_max_unique: int = 20,
    top_n_cat: int = 15,
    show_text_cat: bool = True,
    bargap_cat: float = 0.35,
    annotate_totals_cat: bool = True,
    nbins_num: int = 50,
    opacity_num: float = 0.5,
    density_num: bool = False
) -> go.Figure:
    """
    Una feature per volta:
      - Se 0–5 o categorica: stacked con altezza=conteggi + testo "N (p%)".
      - Se numerica: istogrammi sovrapposti con bins condivisi.
    """
    assert target in df.columns, f"Target '{target}' non trovato."
    s = df[feature]
    if is_rating_0_5(s) or is_categorical(s, max_unique=cat_max_unique):
        return make_categorical_stacked_figure(
            df, feature, target,
            show_text=show_text_cat,
            bargap=bargap_cat,
            top_n_cat=top_n_cat,
            annotate_totals=annotate_totals_cat
        )
    else:
        return make_numeric_hist_overlay_figure(
            df, feature, target,
            nbins=nbins_num,
            opacity=opacity_num,
            density=density_num
        )

### 2.4.1 Gender

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Gender", target="satisfaction")
fig.show()

Guardando il grafico Gender → satisfaction, noto subito due cose: i volumi dei due gruppi sono quasi identici (≈51,2k Male vs ≈52,7k Female) e i tassi di soddisfazione sono molto vicini: 43,9% per i Male e 42,7% per le Female. Nel complesso il dataset si assesta attorno a una soddisfazione media di ~43,3%.

Interpreto così: il genere da solo mostra un segnale debole. C’è una differenza di circa 1,2 punti percentuali a favore dei Male, ma la distanza è piccola.

La mia ipotesi di lavoro è che eventuali differenze apparenti siano in realtà il riflesso di mix di variabili: classe di viaggio, qualità del servizio a bordo, tempi di check-in, ritardi in partenza/arrivo. Il genere potrebbe agire in interazione con questi fattori (classico rischio di effetto di confondimento o di Simpson).

### 2.4.2 Customer Type

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Customer Type", target="satisfaction")
fig.show()

Osservando Customer Type → satisfaction mi colpisce subito il salto netto tra i due gruppi. 

I disloyal Customer sono pochi (≈19k) e hanno un tasso di soddisfazione molto basso (circa 23,7%); i Loyal Customer sono la maggioranza (≈85k) e risultano soddisfatti nel 47,7% dei casi. In pratica, a parità di tutto il resto, l’essere “loyal” raddoppia il tasso di soddisfazione e triplica circa le odds di essere satisfied rispetto ai disloyal. L’altezza diversa delle barre conferma anche il forte sbilanciamento dei volumi: il campione è dominato dai clienti loyal, quindi il modello tenderà a “vedere” soprattutto quel profilo.

“Customer Type” è una variabile molto informativa per la soddisfazione, ma va maneggiata con giudizio. Potrebbe incorporare effetti a monte (es. frequenza di volo, anzianità, servizi accessori) che a loro volta generano soddisfazione. Non è leakage diretto, però è abbastanza vicina al concetto di fedeltà/relazione da rischiare di “schiacciare” il contributo di altre feature se non regolarizzo bene o se non controllo le interazioni.

### 2.4.3 Age

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Age", target="satisfaction", nbins_num=60)
fig.show()

Guardando la distribuzione di Age per classe, vedo che le due curve si sovrappongono quasi completamente: la soddisfazione non sembra dipendere in modo forte dall’età. La massa è concentrata tra i 20 e i 60 anni (normale per un campione di viaggiatori), e a occhio colgo solo scostamenti deboli: nella fascia 35–55 la quota “satisfied” tende a stare leggermente sopra, mentre tra 20–30 e oltre i 65 noto un filo più di “neutral/dissatisfied”. Ma parliamo di differenze sottili che, con istogrammi sovrapposti a conteggio, possono essere influenzate dal volume per fascia.

La mia ipotesi operativa è che Age da sola abbia potere predittivo basso e non lineare (se c’è segnale, è probabilmente a U morbida o con un plateau centrale).

### 2.4.4 Type of Travel

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Type of Travel", target="satisfaction")
fig.show()

Guardando Type of Travel → satisfaction mi viene subito da pensare che qui c’è uno spartiacque vero. I Business travel sono ~71,7k e hanno un tasso di soddisfazione intorno al 58,3%; i Personal travel sono ~32,2k e la soddisfazione scende a circa 10,2%. Tradotto: tra chi viaggia per lavoro la probabilità di essere satisfied è ~5,7× quella dei viaggi personali, e in termini di odds il rapporto è gigantesco (≈ 12:1 a favore del business). Il dato complessivo torna: pesando per i volumi, la soddisfazione media del campione resta attorno al 43%.

**Type of Travel** è una feature molto forte e cattura differenze strutturali del viaggio. I business traveler tendono a interagire con servizi premium o più efficienti (corsie prioritarie, classi superiori, processi più rodati), oppure semplicemente hanno aspettative diverse e valutano l’esperienza in modo più favorevole; al contrario, i personal traveler probabilmente vivono più spesso l’esperienza “economy pura” o subiscono di più l’attrito nei processi. In ogni caso, la distanza di ~48 punti è troppo ampia per essere rumore.

### 2.4.5 Class

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Class", target="satisfaction")
fig.show()

Osservando Class → satisfaction mi faccio questa idea: la classe di viaggio è uno spartiacque fortissimo. In Business vedo circa 69,4% di satisfied (≈49,7k passeggeri), in Eco il tasso crolla a 18,6% (≈46,7k), mentre Eco Plus sta in mezzo, attorno a ~24–25% (≈7,5k). Le barre hanno altezze comparabili per Business ed Eco (campioni grandi), Eco Plus è molto più piccola.

In termini di intensità dell’effetto, il salto è enorme: le odds di essere satisfied in Business sono ~10× quelle di Eco e ~7× quelle di Eco Plus; tra Eco ed Eco Plus c’è un gradino, ma decisamente più contenuto. Questo per me significa che Class da sola spiega una quota importante della variabilità nella soddisfazione.

Non mi fermo alla storia semplice però: so che Class è intrecciata con almeno due fattori chiave — Type of Travel (il business traveler è spesso in Business) e Customer Type (loyal). È probabile che parte del vantaggio di Business rifletta processi più rapidi, servizi prioritari e, in generale, un mix di servizio diverso.

### 2.4.6 Flight Distance

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Flight Distance", target="satisfaction", nbins_num=60)
fig.show()

Guardando la Distribuzione di Flight Distance per classe, mi sembra di leggere una storia abbastanza chiara: sui voli corti (intorno ai 200–800 km) la massa è dominata dai neutral/dissatisfied; man mano che la distanza aumenta (da ~1.500 km in su) la curva dei satisfied prende il sopravvento. In coda, tra i voli molto lunghi (oltre ~2.500–3.000 km), la presenza dei satisfied resta relativamente più alta, pur con volumi totali più bassi (coda lunga).

Come la interpreto: la probabilità di soddisfazione cresce con la distanza, ma non in modo lineare; sembra una curva che parte bassa sui voli brevi, sale tra ~1.000–2.000 km e poi si appiattisce. Quindi “Flight Distance” non è un semplice effetto lineare: mi aspetto non linearità (spline o binning) e interazioni con altre feature. Due ipotesi operative (da verificare, non da prendere per oro colato):

- Sui voli brevi pesa di più l’attrito del processo (code, boarding, tempi morti) rispetto al valore percepito del servizio in volo; un ritardo di 30’ su un volo di 50’ “vale di più” che su un long-haul.
- Sui voli lunghi entrano in gioco comfort, intrattenimento, pasti: c’è più “spazio” perché il servizio faccia la differenza (e spesso cambia anche Class/tariffazione).

### 2.4.7 Inflight wifi service

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Inflight wifi service", target="satisfaction")
fig.show()

Vedo un gradiente quasi monotono: a rating 1–3 la quota di satisfied resta bassa e piatta (~ 25–33%), a 4 c’è un salto netto (~ 60% satisfied) e a 5 il rapporto si ribalta quasi del tutto (~ 99% satisfied). Anche i volumi aiutano a contestualizzare: gli scaglioni 2–3 sono i più popolati (~ 25,8k ciascuno), 4 scende (~ 19,8k) e 5 è più piccolo (~ 11,5k). Questo per me racconta un effetto soglia: la percezione del Wi-Fi “buono davvero” scatta dal 4 in su; sotto, gli utenti lo vivono come non risolutivo.

C’è però un’anomalia vistosa su 0 (≈ 3,1k casi) con ~ 99,7% satisfied: è plausibile che 0 non significhi “pessimo” ma “non applicabile / non usato / non disponibile”. Se chi non ha usato/valutato il Wi-Fi è soddisfatto per altri motivi (classe, puntualità), il 0 diventa una categoria speciale più che il fondo scala.

Nota bene: essendo una valutazione post-volo, questa feature è quasi “gemella” della target. Va benissimo per spiegare la soddisfazione a posteriori o per scoring su survey, ma potrebbe essere leakage se lo use-case è pre-volo (predire la soddisfazione prima di erogare il servizio). In quel caso restringo il set alle variabili disponibili ex-ante o costruisco due modelli (diagnostico vs predittivo operativo).

### 2.4.8 Departure/Arrival time convenient

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Departure/Arrival time convenient", target="satisfaction")
fig.show()

Osservando Departure/Arrival time convenient → satisfaction leggo un pattern controintuitivo. Al crescere del rating di “convenienza” la quota di satisfied scende: siamo ~ 48–49% a 1, ~ 44–43% a 2–3, tocchiamo un minimo ~ 38,8% a 4 e risaliamo appena a 42,5% a 5. Anche i volumi contano: 4 e 5 sono le classi più popolose (~ 25,5k e ~ 22,4k), quindi questa dinamica pesa parecchio sul totale. La categoria 0 (≈ 5,3k) sta a metà (~ 47,5% satisfied) e non sembra un “NA mascherato” come avevo visto sul Wi-Fi.

Cosa ipotizzo mentre lo guardo:

- Ipotesi codifica/inversione: la scala potrebbe essere interpretata al contrario (5 = poco conveniente). Prima di tutto verifico il data dictionary o la distribuzione con Spearman tra rating e 1{satisfied}: mi aspetto una correlazione negativa marcata; se il dizionario dice che 5 è “più conveniente”, allora la variabile è effettivamente anti-correlata con la soddisfazione.
- Ipotesi confondenti: i rating alti di “convenienza” potrebbero concentrarsi su fasce orarie di punta (voli affollati, gate congestionati) o su viaggi personali, dove ho visto tassi di soddisfazione molto più bassi. Anche i ritardi potrebbero essere più frequenti proprio negli slot “comodi”.
- Forma dell’effetto: non è lineare; vedo un declino 1→4 e un piccolo rimbalzo a 5. Potrebbe esserci una U rovesciata o un punto di saturazione nella percezione.

### 2.4.9 Ease of Online booking

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Ease of Online booking", target="satisfaction")
fig.show()

Vedo un pattern quasi monotono: quando il booking online è valutato 1–3 la soddisfazione resta bassa e piatta (~ 30–38%); al valore 4 c’è un salto (circa 53% satisfied) e a 5 il rapporto si ribalta (~ 74% satisfied). I volumi sono alti proprio dove il giudizio è mediocre (2–3, ~24k ciascuno), quindi quel tratto pesa parecchio sulla media complessiva.

Mi insospettisce la categoria 0 (~ 4.5k casi) con ~ 66% satisfied: più che “pessimo”, qui potrebbe voler dire “non applicabile / non usato / canale diverso” (es. prenotazioni tramite agenzia/corporate). Se molti business non usano l’online e sono più soddisfatti per altri motivi (classe, servizi), il 0 diventa una categoria speciale, non il fondo scala.

### 2.4.10 Gate location

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Gate location", target="satisfaction")
fig.show()

La relazione non è monotona. Ai valori 1–2 la quota di satisfied è intorno al 50% → 46%; su 3 tocca il minimo (≈ 34,7%) e risale leggermente a 4 (≈ 38,9%), per poi impennarsi a 5 (≈ 57,2%). I volumi sono alti proprio su 3–4 (≈28,6k e 24,4k), quindi il “buco” centrale pesa parecchio sulla media complessiva. La categoria 0 ha 1 solo record: la considero rumore/codifica anomala, la tratterei come NA separato.

Come la interpreto: la soddisfazione è peggiore nelle valutazioni intermedie (3–4) e migliore agli estremi, soprattutto a 5. Se 5 significa “gate comodo/ben posizionato”, allora ha senso che migliori la percezione complessiva; il crollo su 3–4 potrebbe riflettere gate scomodi, lontani o congestionati (navette, scale mobili affollate, corridoi lunghi). Potrebbe anche esserci un effetto aeroporto/terminal: alcuni hub assegnano sistematicamente gate “peggiori” a certi voli.

### 2.4.11 Food and drink

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Food and drink", target="satisfaction")
fig.show()

Guardando Food and drink → satisfaction mi viene naturale leggere un gradiente crescente: ai voti 1–3 la quota di satisfied rimane bassa e stabile (≈ 20% a 1, 39% a 2–3), poi a 4 c’è il cambio di regime (≈ 52,5%) e a 5 si consolida (≈ 54,9%). I volumi sono alti proprio tra 2–5 (≈22–24k ciascuno), quindi questo pattern pesa molto sul totale. La categoria 0 (n=107) è trascurabile: la tratto come rumore o “non applicabile”.

Come la interpreto da analista:
- C’è un effetto soglia chiaro tra 3→4: è lì che la percezione del catering “diventa abbastanza buona da spostare la soddisfazione oltre il 50%”.
- Tra 4→5 vedo rendimenti decrescenti: il miglioramento c’è, ma è più piccolo.
- Sotto il cofano potrebbero agire Class e Type of Travel (es. Business con catering migliore; Business traveler più indulgenti), quindi parte del gradiente potrebbe riflettere il mix di prodotto più che il solo cibo.

### 2.4.12 Online boarding

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Online boarding", target="satisfaction")
fig.show()

Guardando Online boarding → satisfaction mi viene fuori un segnale pulitissimo e… rumorosamente monotono.

Vedo che ai voti 0–3 la quota di satisfied è molto bassa (intorno al 10–14%; per 3 leggo ~ 13,6%). Poi, al valore 4 c’è il cambio di fase: si sale a ~ 62,3% satisfied, e a 5 si arriva addirittura a ~ 87,2%. Anche i volumi contano: 4 è lo scaglione più popoloso (~ 30,8k), 5 è grande (~ 20,7k), mentre 0–1–2–3 sono sensibilmente più piccoli. In pratica: quando l’esperienza di online boarding è percepita come “buona” (≥4), la soddisfazione si ribalta.

Interpreto così: c’è un effetto soglia netto tra 3→4. Sotto quella soglia l’esperienza non “sblocca” valore percepito; sopra, sì — e il passaggio a 5 porta un ulteriore salto, coerente con un processo davvero fluido (app che funziona, QR immediato, poche frizioni ai varchi). La categoria 0 (≈2,4k) con soddisfazione bassissima probabilmente non è “peggiore del peggiore”, ma “non applicabile / non usato / impossibilitato a fare l’online boarding”; la tratto come flag separato, non come fondo della scala.

Sicuramente rendere l’online boarding “almeno buono” (4) è un punto di leva; l’upgrade 4→5 aggiunge ulteriore valore, ma il grosso della curva si vince superando la soglia del 3. Operativamente: ridurre attriti (login, documenti, sincronizzazione PNR, QR stabile offline), chiarezza delle eccezioni e integrazione con il check-in.

### 2.4.13 Seat comfort

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Seat comfort", target="satisfaction")
fig.show()

Qui c’è un driver forte con un chiaro effetto soglia.

Vedo che ai voti 1–3 la quota di satisfied resta piatta e bassa (≈21–22%: 1→22,3%, 2→22,4%, 3→21,2%) con volumi non trascurabili (≈12k, 15k, 18,7k). Dal 3 al 4 cambia il film: il satisfied balza a 56,1% (≈31,8k record, lo scaglione più popoloso), e a 5 arriva a 65,0% (≈26,5k). La categoria 0 è un singolo caso: la tratto come rumore/NA.

La comodità del sedile sembra essere una leva concreta di soddisfazione. Non vedo una crescita lineare: è più un salto 3→4 (≈+35 punti percentuali) e poi rendimenti decrescenti 4→5 (≈+9 pp). Mi aspetto però confondenti importanti: soprattutto Class (Business ↔ sedili migliori) e Type of Travel. Parte del gradiente potrebbe riflettere il mix di cabina/servizi più che il solo sedile.

Nota bene: essendo un rating post-volo, Seat comfort è ottimo per spiegare la soddisfazione o per scoring su survey; per un uso predittivo pre-erogazione rischio leakage. In quel caso separo chiaramente il modello “diagnostico” (con i rating) da quello “operativo” (solo variabili note ex-ante).

### 2.4.14 Inflight entertainment

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Inflight entertainment", target="satisfaction")
fig.show()

Guardando Inflight entertainment → satisfaction mi sembra di vedere un pattern pulito e molto “didattico”. Fino a 3 la soddisfazione resta bassa e quasi piatta (≈ 14% a 1, 21% a 2, 27% a 3 con volumi importanti: ~ 12,5k, 17,6k, 19,1k). Dal 3 al 4 cambia marcia: saliamo a ~ 61% satisfied (≈ 29,4k casi, il cluster più popoloso) e a 5 ci attestiamo su ~65% (≈ 25,2k). La categoria 0 ha 14 record—la tratto come rumore/NA.

Cosa ne deduco: l’intrattenimento di bordo è un driver forte con effetto soglia 3→4 e rendimenti decrescenti 4→5. Non mi aspetto linearità; la relazione è chiaramente ordinale e crescente. È anche plausibile un effetto di contesto: su tratte più lunghe (o in Business) l’entertainment pesa e spesso è migliore. Quindi parte del segnale potrebbe riflettere Class e Flight Distance oltre che la qualità intrinseca del servizio.

### 2.4.15 On-board service

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="On-board service", target="satisfaction")
fig.show()

Vedo che ai voti 1–3 la soddisfazione resta bassa e quasi piatta (~ 19% → 26% → 32%), poi dal 3 al 4 c’è il cambio di fase (~ 54% satisfied, ed è anche lo scaglione più popoloso, ~ 30,9k). A 5 si consolida (~ 64%). La categoria 0 ha 3 record: la considero rumore/NA e non la mescolo alla scala.

Come lo interpreto: il servizio a bordo “funziona davvero” solo oltre la soglia 3→4 (salto ~ +22pp), mentre 4→5 aggiunge valore ma con rendimenti decrescenti (~ +10pp). In termini di intensità, il confronto 3 vs 5 raddoppia circa le odds di essere satisfied (ordine di grandezza ~ 4×), quindi non è un segnale cosmetico: pesa.

### 2.4.16 Leg room service

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Leg room service", target="satisfaction")
fig.show()

Ai voti 1–3 la soddisfazione resta bassa e piatta (≈ 20–27%), poi dal 3 al 4 c’è il salto vero: arrivo a ~ 58,2% satisfied, e a 5 si stabilizza su ~ 61,4%. I volumi sono consistenti proprio su 3–5 (≈ 20k, 28,8k, 24,7k), quindi questo gradino pesa parecchio sul totale. La classe 0 è rara (≈ 472 casi): la considero un’anomalia/NA e non la mescolo alla scala.

La spaziosità per le gambe è un driver forte della soddisfazione, ma l’effetto non è lineare. 

Vedo un effetto soglia 3→4 (il “salto di qualità”), poi rendimenti decrescenti 4→5. So però che qui può giocare il mix di cabina: più Business ⇒ più legroom ⇒ più soddisfazione. Quindi parte del gradiente potrebbe riflettere Class (e anche Type of Travel e Flight Distance) oltre al solo servizio “leg room”.

### 2.4.17 Baggage handling

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Baggage handling", target="satisfaction")
fig.show()

Vedo un profilo quasi monotono crescente, con un punto debole a 3. Sui voti 1–2 la quota di satisfied è simile (≈ 30%), scende a 3 (≈ 23,6%), poi salta a 4 (≈ 48,1%) e si rafforza a 5 (≈ 61,3%). In termini di volumi, il grosso sta proprio su 4 (~ 37,4k) e 5 (~ 27,1k), quindi quel salto 3→4 pesa molto sulla media complessiva. La categoria 0 non compare (0 record), nessun problema di codifica qui.

La gestione bagagli è un driver importante della soddisfazione. Il fatto che 3 sia peggiore di 1–2 mi suggerisce che “3 = esperienza mediocre” irrita più di un giudizio apertamente negativo (paradosso del “meh”): attese irregolari al nastro, comunicazioni poco chiare, piccole frizioni che si sommano. La soglia operativa è 3→4 (≈ +24,5 pp), mentre 4→5 porta ancora beneficio (≈ +13,2 pp), ma con rendimenti decrescenti. Nel complesso, tra 3 e 5 il delta è enorme (≈ +37,7 pp).

### 2.4.18 Checkin service

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Checkin service", target="satisfaction")
fig.show()

Vedo una crescita netta con un salto importante tra 2→3: a 1–2 la soddisfazione è bassa (~ 24–25%), a 3 balza a ~ 44,8% e a 4 si assesta ~ 46%; il picco è a 5 con ~ 61%. Anche i volumi contano: gli scaglioni più popolosi sono 3–4 (~ 28,4k e ~ 29,1k), mentre 5 è comunque consistente (~ 20,6k). La categoria 0 è un record singolo: la tratto come anomalia/NA.

Come lo interpreto: il check-in è un driver forte ma con effetto soglia. Finché l’esperienza è insufficiente/mediocre (1–2), la soddisfazione resta depressa; superata la soglia 3, il rapporto si ribalta, con ulteriori guadagni fino a 5 (anche se 3→4 cresce poco, il vero cambio di fase è 2→3). Mi aspetto confondenti: Class e Type of Travel (corsie/business), e soprattutto correlazione con Online boarding; processi digitali fluidi tendono a migliorare entrambi i rating.

### 2.4.19 Inflight service

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Inflight service", target="satisfaction")
fig.show()

Nei voti 1–3 la soddisfazione resta bassa (~ 24–30%) e, curiosamente, 3 è il peggiore (~ 23,9%). 

Dal 3 al 4 c’è il vero cambio di fase: salgo a ~ 47,8% (ed è anche lo scaglione più popoloso, ~ 37,9k), poi a 5 arrivo a ~ 61%. La categoria 0 ha 3 casi: la considero rumore/NA.

Come la interpreto: l’inflight service è un driver forte con effetto soglia 3→4, poi rendimenti decrescenti 4→5. Il “buco” a 3 (peggio di 2) mi fa pensare a quell’esperienza mediocre che irrita più di un giudizio apertamente negativo: tempi di risposta, cortesia altalenante, annunci poco chiari. So anche che questo rating si muove insieme ad altri (Seat comfort, Food & drink, Entertainment) e con Class/Type of Travel: parte del gradiente potrebbe riflettere il mix di cabina più che il solo servizio di bordo.

### 2.4.20 Cleanliness

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Cleanliness", target="satisfaction")
fig.show()

Guardando Cleanliness → satisfaction leggo un pattern pulito e crescente. Sui voti 1–2 la soddisfazione è bassa (~ 19,6% e ~ 21,3%), poi a 3 salta al ~ 43,2% (praticamente in linea con la media complessiva che ho visto finora), a 4 sale a ~ 53,5% e a 5 arriva a ~ 60,9%. 

I volumi sono importanti proprio dove il servizio è percepito meglio (3–5: ~ 24,6k, ~ 27,2k, ~ 22,7k), quindi questo gradiente pesa davvero sul totale. La classe 0 ha 12 casi: la tratto come anomalia/NA, non la includo nella scala.

Come la interpreto: la pulizia è un driver chiaro con effetto soglia. Passare da un’esperienza scarsa (1–2) a 3 riporta già la soddisfazione verso la baseline; la vera spinta arriva poi da 3→4 (+10pp) e si consolida 4→5 (+7pp), con rendimenti decrescenti nell’ultimo tratto. Mi aspetto che il segnale si intrecci con Class e con altri rating di servizio (On-board, Food & drink, Seat comfort): ambienti più curati e spaziosi coincidono spesso con cabine migliori e processi più fluidi.

### 2.4.21 Departure Delay in Minutes

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Departure Delay in Minutes", target="satisfaction", nbins_num=60)
fig.show()

La variabile è estremamente asimmetrica a destra. C’è una grossa massa vicino allo zero (molti voli senza ritardo o con pochi minuti) e una coda lunghissima che arriva a centinaia di minuti. 

Le due curve (satisfied vs non satisfied) si sovrappongono nei micro-ritardi, ma appena mi allontano dallo zero vedo che la quota di non satisfied cresce: gli istogrammi lo suggeriscono, anche se la scala logica lo nasconde un po’ perché il picco iniziale domina.

Le mie deduzioni:
- La variabile ha una chiara zero-inflation (tanti 0) e outlier pesanti. Così com’è, in un modello lineare rischia di comportarsi male.
- L’effetto sulla soddisfazione non è binario “ritardo sì/no”, è più da soglie: attorno ai 15 minuti (soglia operativa classica), poi 30, 60 e oltre. Oltre queste soglie mi aspetto un crollo progressivo della soddisfazione.
- L’istogramma a conteggi non permette di leggere bene i tassi; serve passare a una vista per proporzioni.

### 2.4.22 Arrival Delay in Minutes

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Arrival Delay in Minutes", target="satisfaction", nbins_num=60)
fig.show()

Guardando la Distribuzione di Arrival Delay in Minutes per classe, mi ritrovo la stessa storia del departure ma più “sensibile” lato cliente: forte asimmetria a destra, tanta massa a 0–5′ e una coda lunghissima che spinge fino a centinaia di minuti. Le due curve si sovrappongono quasi del tutto vicino allo zero; appena i minuti crescono, la porzione di neutral/dissatisfied prende il sopravvento. Non sorprende: ciò che ricordo davvero è quando arrivo, non quando parto.

Come la interpreto:
- Variabile zero-inflated e con outlier pesanti; a conteggi l’effetto si “schiaccia” sul picco iniziale, ma il trend è chiaro: più ritardo all’arrivo ⇒ meno soddisfazione.
- L’impatto è a soglie più che lineare: mi aspetto rotture nette attorno a 15′, 30′, 60′.
- Rispetto al departure, l’arrival delay dovrebbe avere potere predittivo maggiore sulla soddisfazione (è il “dolore” finale).

## 2.5 Outliers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats  # Import mancante!
import warnings
warnings.filterwarnings('ignore')

# Identifica colonne numeriche continue (non rating 0-5)
continuous_cols = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']

# Metodo 1: IQR (Interquartile Range)
def detect_outliers_iqr(df, columns):
    outliers_summary = {}
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
        outliers_count = outliers_mask.sum()
        outliers_percentage = (outliers_count / len(df)) * 100
        
        outliers_summary[col] = {
            'count': outliers_count,
            'percentage': outliers_percentage,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound
        }
    
    return outliers_summary

# Metodo 2: Z-Score
def detect_outliers_zscore(df, columns, threshold=3):
    outliers_summary = {}
    
    for col in columns:
        # Gestisci eventuali NaN
        col_data = df[col].dropna()
        z_scores = np.abs(stats.zscore(col_data))
        outliers_count = (z_scores > threshold).sum()
        outliers_percentage = (outliers_count / len(df)) * 100
        
        outliers_summary[col] = {
            'count': outliers_count,
            'percentage': outliers_percentage,
            'threshold': threshold
        }
    
    return outliers_summary

# Applica entrambi i metodi su raw_data_train
outliers_iqr = detect_outliers_iqr(raw_data_train, continuous_cols)
outliers_zscore = detect_outliers_zscore(raw_data_train, continuous_cols)

# Crea un DataFrame riassuntivo
outliers_results = pd.DataFrame({
    'IQR_count': [outliers_iqr[col]['count'] for col in continuous_cols],
    'IQR_%': [outliers_iqr[col]['percentage'] for col in continuous_cols],
    'Z-Score_count': [outliers_zscore[col]['count'] for col in continuous_cols],
    'Z-Score_%': [outliers_zscore[col]['percentage'] for col in continuous_cols]
}, index=continuous_cols)

print("OUTLIER ANALYSIS RESULTS:")
print("="*60)
print(outliers_results.round(2))

# Visualizzazione
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(continuous_cols):
    ax = axes[idx]
    
    # Box plot con outlier evidenziati
    box_data = raw_data_train[col].dropna()
    box = ax.boxplot(box_data, vert=True, patch_artist=True)
    box['boxes'][0].set_facecolor('lightblue')
    box['medians'][0].set_color('red')
    
    ax.set_title(f'{col}\nOutliers IQR: {outliers_iqr[col]["percentage"]:.1f}%')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Analisi aggiuntiva: mostra i valori estremi per ogni variabile
print("\n" + "="*60)
print("ANALISI DETTAGLIATA OUTLIERS:")
print("="*60)

for col in continuous_cols:
    print(f"\n{col}:")
    print("-" * 30)
    
    # Statistiche base
    col_data = raw_data_train[col].dropna()
    print(f"  Min: {col_data.min():.2f}")
    print(f"  Q1: {col_data.quantile(0.25):.2f}")
    print(f"  Mediana: {col_data.median():.2f}")
    print(f"  Q3: {col_data.quantile(0.75):.2f}")
    print(f"  Max: {col_data.max():.2f}")
    
    # Bounds IQR
    print(f"  IQR Lower Bound: {outliers_iqr[col]['lower_bound']:.2f}")
    print(f"  IQR Upper Bound: {outliers_iqr[col]['upper_bound']:.2f}")
    
    # Percentuale outliers
    print(f"  Outliers (IQR): {outliers_iqr[col]['count']} ({outliers_iqr[col]['percentage']:.2f}%)")
    print(f"  Outliers (Z-score>3): {outliers_zscore[col]['count']} ({outliers_zscore[col]['percentage']:.2f}%)")

**Decisione operativa sugli outlier.**

L'analisi mostra due regimi molto diversi:

- **Age** e **Flight Distance** hanno una percentuale di outlier marginale (≤2.2% con IQR, ≤0.06% con Z-score) e i valori estremi sono comunque plausibili (passeggeri fino a 85 anni, voli fino a ~5000 km). Li mantengo senza modifiche.
- **Departure Delay** e **Arrival Delay** mostrano invece il 13–14% di outlier IQR, con una coda lunghissima fino a ~1600 minuti. La distribuzione è zero-inflated e asimmetrica: rimuoverli significherebbe perdere proprio i casi più informativi per la soddisfazione (un volo con 6 ore di ritardo è un caso d'uso reale, non rumore).

Scelta operativa: **non rimuovo** alcun outlier perché i modelli finali saranno tree-based (Random Forest, Gradient Boosting, AdaBoost) che sono naturalmente robusti alle code pesanti. Per neutralizzare comunque l'effetto distorsivo degli outlier sui modelli che assumono distribuzioni più regolari (Logistic Regression, usata come baseline nello spot-check), applicherò più avanti **RobustScaler** alle due variabili di ritardo e **StandardScaler** alle altre numeriche, anziché un unico scaler globale come inizialmente ipotizzato.

## 2.6 Analisi delle Variabili Categoriche

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Identifica variabili categoriche
categorical_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

print("CATEGORICAL VARIABLES ANALYSIS:")
print("="*60)

# Analisi dettagliata per ogni variabile categorica
for col in categorical_cols:
    print(f"\n{col}:")
    print("-" * 30)
    
    # Conteggi e percentuali
    value_counts = raw_data_train[col].value_counts()
    value_percentages = raw_data_train[col].value_counts(normalize=True) * 100
    
    cat_summary = pd.DataFrame({
        'Count': value_counts,
        'Percentage': value_percentages.round(2)
    })
    
    print(cat_summary)
    print(f"Unique values: {raw_data_train[col].nunique()}")
    print(f"Missing values: {raw_data_train[col].isnull().sum()}")

print("\n" + "="*60)
print("STRATEGIA DI ENCODING CONSIGLIATA:")
print("="*60)

# Determina strategia di encoding basata sul numero di categorie
encoding_strategy = {}
for col in categorical_cols:
    n_unique = raw_data_train[col].nunique()
    if n_unique == 2:
        encoding_strategy[col] = "Label Encoding (Binary)"
    elif n_unique <= 5:
        encoding_strategy[col] = "One-Hot Encoding"
    else:
        encoding_strategy[col] = "Target Encoding or Embedding"
    
    print(f"{col}: {n_unique} categorie → {encoding_strategy[col]}")

print("\n" + "="*60)
print("IMPLEMENTAZIONE ENCODING:")
print("="*60)

# Crea una copia per preservare i dati originali
raw_data_encoded = raw_data_train.copy()

# 1. LABEL ENCODING per variabili binarie
binary_cols = [col for col in categorical_cols if raw_data_train[col].nunique() == 2]
le_dict = {}  # Dizionario per salvare i LabelEncoder per uso futuro

print("\n1. LABEL ENCODING (Variabili Binarie):")
print("-" * 30)

for col in binary_cols:
    le = LabelEncoder()
    raw_data_encoded[f'{col}_encoded'] = le.fit_transform(raw_data_train[col])
    le_dict[col] = le  # Salva l'encoder
    
    # Mostra il mapping
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"\n{col}:")
    for original, encoded in mapping.items():
        count = (raw_data_train[col] == original).sum()
        pct = (count / len(raw_data_train)) * 100
        print(f"  '{original}' → {encoded} (n={count}, {pct:.1f}%)")

# 2. ONE-HOT ENCODING per variabili multi-classe
multi_class_cols = [col for col in categorical_cols if raw_data_train[col].nunique() > 2]

print("\n2. ONE-HOT ENCODING (Variabili Multi-classe):")
print("-" * 30)

for col in multi_class_cols:
    print(f"\n{col} ({raw_data_train[col].nunique()} categorie):")
    
    # Crea dummies
    dummies = pd.get_dummies(raw_data_train[col], prefix=col, drop_first=False)
    
    # Mostra le nuove colonne create
    print(f"  Nuove colonne create: {list(dummies.columns)}")
    
    # Statistiche per ogni dummy
    for dummy_col in dummies.columns:
        count = dummies[dummy_col].sum()
        pct = (count / len(raw_data_train)) * 100
        print(f"    {dummy_col}: {count} records ({pct:.1f}%)")
    
    # Aggiungi al dataframe encoded (opzionale, per dimostrazione)
    raw_data_encoded = pd.concat([raw_data_encoded, dummies], axis=1)

# 3. ANALISI DELL'IMPATTO SULL'ESPANSIONE DELLE FEATURES
print("\n" + "="*60)
print("IMPATTO DELL'ENCODING SUL DATASET:")
print("="*60)

original_cols = len(raw_data_train.columns)
after_encoding_cols = len(raw_data_encoded.columns)
new_cols = after_encoding_cols - original_cols

print(f"Colonne originali: {original_cols}")
print(f"Colonne dopo encoding: {after_encoding_cols}")
print(f"Nuove colonne create: {new_cols}")

# 4. VERIFICA CORRELAZIONE TRA CATEGORICHE E TARGET
print("\n" + "="*60)
print("CORRELAZIONE CATEGORICHE CON TARGET:")
print("="*60)

from scipy.stats import chi2_contingency

for col in categorical_cols:
    # Crea tabella di contingenza
    contingency_table = pd.crosstab(raw_data_train[col], raw_data_train['satisfaction'])
    
    # Test Chi-quadrato
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    # Calcola Cramér's V per misurare la forza dell'associazione
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim))
    
    print(f"\n{col}:")
    print(f"  Chi-square: {chi2:.2f}")
    print(f"  P-value: {p_value:.4f}")
    print(f"  Cramér's V: {cramers_v:.3f}", end=" ")
    
    # Interpretazione della forza dell'associazione
    if cramers_v < 0.1:
        strength = "(molto debole)"
    elif cramers_v < 0.3:
        strength = "(debole)"
    elif cramers_v < 0.5:
        strength = "(moderata)"
    else:
        strength = "(forte)"
    print(strength)

# 5. VISUALIZZAZIONE DISTRIBUZIONE CATEGORICHE PER TARGET
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(categorical_cols):
    ax = axes[idx]
    
    # Crea DataFrame per plotting
    plot_data = raw_data_train.groupby([col, 'satisfaction']).size().unstack()
    
    # Calcola percentuali per riga
    plot_data_pct = plot_data.div(plot_data.sum(axis=1), axis=0) * 100
    
    # Crea stacked bar plot
    plot_data_pct.plot(kind='bar', stacked=True, ax=ax, 
                       color=['coral', 'lightblue'],
                       edgecolor='black', linewidth=0.5)
    
    ax.set_title(f'{col} vs Satisfaction', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Percentage (%)')
    ax.legend(title='Satisfaction', loc='best')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    
    # Aggiungi percentuali sulle barre
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', label_type='center')

plt.suptitle('Distribuzione Variabili Categoriche per Satisfaction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 6. RACCOMANDAZIONI FINALI
print("\n" + "="*60)
print("RACCOMANDAZIONI PER IL TRATTAMENTO DELLE CATEGORICHE:")
print("="*60)

print("\n1. ENCODING STRATEGY:")
for col in categorical_cols:
    if col in binary_cols:
        print(f"   • {col}: Usa Label Encoding (binaria)")
    else:
        print(f"   • {col}: Usa One-Hot Encoding ({raw_data_train[col].nunique()} categorie)")

print("\n2. FEATURE IMPORTANCE:")
important_cats = []
for col in categorical_cols:
    contingency_table = pd.crosstab(raw_data_train[col], raw_data_train['satisfaction'])
    chi2, p_value, _, _ = chi2_contingency(contingency_table)
    if p_value < 0.001:
        important_cats.append(col)

print(f"   Variabili categoriche significative (p<0.001): {important_cats}")

In [ ]:
raw_data_encoded

## 2.7 Gestione dello "0" nei rating come valore "non applicabile"

Durante l'EDA feature-by-feature ho notato un pattern ricorrente: per diverse rating (Inflight wifi service, Ease of Online booking, Departure/Arrival time convenient, ecc.) il valore `0` non si comporta come il fondo scala ("pessimo"), ma come una categoria a sé: spesso il tasso di soddisfazione per chi ha votato 0 è anomalo (es. Wi-Fi service: 99.7% satisfied a rating 0). L'interpretazione più sensata è che `0 = "non applicabile / non utilizzato / non disponibile"`.

Lasciare quei `0` mescolati nella scala 0–5 distorce sia lo scaling sia i test statistici. Procedo così:

1. per ogni rating che contiene almeno un `0`, creo una **flag binaria** `<feature>_zeroflag` che marca i casi "non applicabile";
2. sostituisco lo `0` con `NaN` e poi lo imputo con la **mediana del training set** (quella vera, calcolata sui rating effettivi 1–5);
3. tengo separatamente l'informazione "non applicabile" come feature binaria, così il modello può imparare se quella categoria ha un peso autonomo.

In [ ]:
# ============================================================
# 2.7 - GESTIONE DEI RATING == 0 COME "NON APPLICABILE"
# ============================================================
print("GESTIONE RATING == 0")
print("=" * 60)

# Lista dei rating su scala 0-5 (tutti quelli che ho ispezionato nell'EDA)
rating_features_all = [
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness'
]

# Identifico solo le feature per cui lo zero è plausibilmente "non applicabile":
# tengo quelle che hanno almeno un record con valore == 0
zero_features = [c for c in rating_features_all if (raw_data_train[c] == 0).any()]

print(f"Feature con almeno un rating == 0: {len(zero_features)}")
for c in zero_features:
    n_zero = int((raw_data_train[c] == 0).sum())
    pct = n_zero / len(raw_data_train) * 100
    print(f"  {c:40s} -> {n_zero:>5d} zeri ({pct:.2f}%)")

# Creo una versione "pulita" del training set:
#   1) flag binarie *_zeroflag
#   2) 0 -> NaN -> imputazione con mediana dei rating 1-5
raw_data_train_clean = raw_data_train.copy()
zero_flag_cols = []
median_imputation_map = {}

for c in zero_features:
    flag_name = f"{c}_zeroflag"
    raw_data_train_clean[flag_name] = (raw_data_train_clean[c] == 0).astype(int)
    zero_flag_cols.append(flag_name)

    # mediana calcolata SOLO sui valori 1-5 del training set
    median_val = raw_data_train_clean.loc[raw_data_train_clean[c] > 0, c].median()
    median_imputation_map[c] = median_val

    raw_data_train_clean[c] = raw_data_train_clean[c].replace(0, np.nan)
    raw_data_train_clean[c] = raw_data_train_clean[c].fillna(median_val)

print("\nNuove flag create:")
print(zero_flag_cols)

print("\nMappa di imputazione (mediana 1-5 del train):")
for c, v in median_imputation_map.items():
    print(f"  {c:40s} -> {v}")

# Verifica: nessun rating dovrebbe valere 0 nel dataset pulito
remaining_zeros = sum((raw_data_train_clean[c] == 0).sum() for c in zero_features)
print(f"\nRating con valore 0 dopo la pulizia: {remaining_zeros}")


## 2.8 Standardizzazione delle Features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
import warnings
warnings.filterwarnings('ignore')

print("STANDARDIZZAZIONE DELLE FEATURES")
print("="*60)

# 1. IDENTIFICAZIONE DELLE FEATURES DA SCALARE
print("\n1. IDENTIFICAZIONE FEATURES:")
print("-" * 30)

# Identifica diversi tipi di features
# a) Features numeriche continue
continuous_features = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']

# b) Features di rating (0-5)
rating_features = [col for col in raw_data_train.columns 
                  if any(keyword in col.lower() for keyword in 
                  ['service', 'comfort', 'boarding', 'handling', 
                   'cleanliness', 'entertainment', 'wifi', 'convenient', 
                   'booking', 'location', 'food'])]

# c) Features categoriche (da NON scalare)
categorical_features = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

# d) Features da escludere
exclude_features = ['id', 'satisfaction']

# Features totali da scalare
features_to_scale = continuous_features + rating_features

print(f"Features continue: {len(continuous_features)}")
print(f"  → {continuous_features}")
print(f"\nFeatures rating (0-5): {len(rating_features)}")
print(f"  → {rating_features}")
print(f"\nFeatures categoriche (NON scalare): {len(categorical_features)}")
print(f"  → {categorical_features}")
print(f"\nTotale features da scalare: {len(features_to_scale)}")

# 2. ANALISI PRE-SCALING
print("\n" + "="*60)
print("2. ANALISI DISTRIBUZIONE PRE-SCALING:")
print("-" * 30)

# Statistiche descrittive prima dello scaling
pre_scaling_stats = raw_data_train[features_to_scale].describe()

print("\nStatistiche principali (subset di features):")
sample_features = continuous_features + rating_features[:2]  # Mostra solo alcune per leggibilità

stats_summary = pd.DataFrame({
    'Mean': raw_data_train[sample_features].mean(),
    'Std': raw_data_train[sample_features].std(),
    'Min': raw_data_train[sample_features].min(),
    'Max': raw_data_train[sample_features].max(),
    'Range': raw_data_train[sample_features].max() - raw_data_train[sample_features].min()
})

print(stats_summary.round(2))

# Identificazione problemi di scala
print("\n" + "="*60)
print("3. PROBLEMI DI SCALA IDENTIFICATI:")
print("-" * 30)

# Calcola il rapporto tra i range
ranges = raw_data_train[features_to_scale].max() - raw_data_train[features_to_scale].min()
max_range = ranges.max()
min_range = ranges.min()

print(f"Range massimo: {max_range:.2f} (feature: {ranges.idxmax()})")
print(f"Range minimo: {min_range:.2f} (feature: {ranges.idxmin()})")
print(f"Rapporto max/min range: {max_range/min_range:.1f}x")

if max_range/min_range > 10:
    print("⚠️ FORTE DIFFERENZA DI SCALA - Standardizzazione NECESSARIA!")
else:
    print("✓ Differenze di scala moderate")

# 3. CONFRONTO TRA METODI DI SCALING
print("\n" + "="*60)
print("4. CONFRONTO METODI DI SCALING:")
print("-" * 30)

# Crea copie del dataset per diversi metodi
data_standard = raw_data_train.copy()
data_minmax = raw_data_train.copy()
data_robust = raw_data_train.copy()

# Gestisci valori mancanti prima dello scaling
print("\nGestione valori mancanti prima dello scaling:")
missing_in_scale = raw_data_train[features_to_scale].isnull().sum()
if missing_in_scale.sum() > 0:
    print(f"  Trovati {missing_in_scale.sum()} valori mancanti")
    print("  → Imputazione con mediana per robustezza")
    for feature in features_to_scale:
        median_val = raw_data_train[feature].median()
        data_standard[feature] = data_standard[feature].fillna(median_val)
        data_minmax[feature] = data_minmax[feature].fillna(median_val)
        data_robust[feature] = data_robust[feature].fillna(median_val)
else:
    print("  ✓ Nessun valore mancante nelle features da scalare")

# Applica diversi scaler
scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()
scaler_robust = RobustScaler()

data_standard[features_to_scale] = scaler_standard.fit_transform(data_standard[features_to_scale])
data_minmax[features_to_scale] = scaler_minmax.fit_transform(data_minmax[features_to_scale])
data_robust[features_to_scale] = scaler_robust.fit_transform(data_robust[features_to_scale])

# Confronto risultati
print("\n" + "="*60)
print("5. RISULTATI SCALING:")
print("-" * 30)

scaling_comparison = pd.DataFrame({
    'Original_Mean': raw_data_train[features_to_scale].mean(),
    'Original_Std': raw_data_train[features_to_scale].std(),
    'Standard_Mean': data_standard[features_to_scale].mean(),
    'Standard_Std': data_standard[features_to_scale].std(),
    'MinMax_Min': data_minmax[features_to_scale].min(),
    'MinMax_Max': data_minmax[features_to_scale].max(),
    'Robust_Median': data_robust[features_to_scale].median()
})

print("\nConfronto metodi (prime 5 features):")
print(scaling_comparison.head().round(3))

# Verifica qualità dello StandardScaler
print("\n" + "="*60)
print("6. VERIFICA STANDARD SCALER:")
print("-" * 30)

mean_after = data_standard[features_to_scale].mean().mean()
std_after = data_standard[features_to_scale].std().mean()

print(f"Media globale dopo StandardScaler: {mean_after:.6f} (atteso: ~0)")
print(f"Deviazione standard media dopo StandardScaler: {std_after:.6f} (atteso: ~1)")

if abs(mean_after) < 0.001 and abs(std_after - 1) < 0.01:
    print("✓ StandardScaler applicato correttamente!")
else:
    print("⚠️ Verificare l'applicazione dello StandardScaler")

# 4. VISUALIZZAZIONE DISTRIBUZIONE PRE E POST SCALING
print("\n" + "="*60)
print("7. VISUALIZZAZIONE IMPATTO SCALING:")
print("-" * 30)

# Seleziona features rappresentative per la visualizzazione
viz_features = ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure Delay in Minutes']

fig, axes = plt.subplots(len(viz_features), 3, figsize=(15, len(viz_features)*3))

for idx, feature in enumerate(viz_features):
    # Original
    ax1 = axes[idx, 0]
    raw_data_train[feature].dropna().hist(bins=30, ax=ax1, color='lightblue', edgecolor='black')
    ax1.set_title(f'Original\nMean: {raw_data_train[feature].mean():.2f}, Std: {raw_data_train[feature].std():.2f}')
    ax1.set_xlabel(feature)
    ax1.set_ylabel('Frequency')
    
    # StandardScaler
    ax2 = axes[idx, 1]
    data_standard[feature].hist(bins=30, ax=ax2, color='lightgreen', edgecolor='black')
    ax2.set_title(f'StandardScaler\nMean: {data_standard[feature].mean():.2f}, Std: {data_standard[feature].std():.2f}')
    ax2.set_xlabel(f'{feature} (scaled)')
    
    # RobustScaler
    ax3 = axes[idx, 2]
    data_robust[feature].hist(bins=30, ax=ax3, color='lightsalmon', edgecolor='black')
    ax3.set_title(f'RobustScaler\nMedian: {data_robust[feature].median():.2f}')
    ax3.set_xlabel(f'{feature} (scaled)')

plt.suptitle('Confronto Metodi di Scaling', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 5. ANALISI IMPATTO OUTLIER SULLO SCALING
print("\n" + "="*60)
print("8. IMPATTO OUTLIER SULLO SCALING:")
print("-" * 30)

# Confronta scaling su features con molti outlier (Delay features)
outlier_features = ['Departure Delay in Minutes', 'Arrival Delay in Minutes']

for feature in outlier_features:
    if feature in features_to_scale:
        original_data = raw_data_train[feature].dropna()
        
        # Calcola percentuale di outlier (IQR method)
        Q1 = original_data.quantile(0.25)
        Q3 = original_data.quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((original_data < Q1 - 1.5*IQR) | (original_data > Q3 + 1.5*IQR)).sum()
        outlier_pct = (outliers / len(original_data)) * 100
        
        print(f"\n{feature}:")
        print(f"  Outlier: {outlier_pct:.1f}%")
        print(f"  StandardScaler range: [{data_standard[feature].min():.2f}, {data_standard[feature].max():.2f}]")
        print(f"  RobustScaler range: [{data_robust[feature].min():.2f}, {data_robust[feature].max():.2f}]")
        
        if outlier_pct > 5:
            print(f"  → Raccomandazione: RobustScaler (resistente a outlier)")
        else:
            print(f"  → Raccomandazione: StandardScaler (distribuzione normale)")



# Applica effettivamente lo StandardScaler al dataset
print("\n" + "="*60)
print("APPLICAZIONE FINALE:")
print("-" * 30)

# Crea il dataset finale scalato
raw_data_train_scaled = raw_data_train.copy()

# Imputa valori mancanti con mediana
for feature in features_to_scale:
    if raw_data_train_scaled[feature].isnull().sum() > 0:
        median_val = raw_data_train_scaled[feature].median()
        raw_data_train_scaled[feature] = raw_data_train_scaled[feature].fillna(median_val)

# Applica StandardScaler
final_scaler = StandardScaler()
raw_data_train_scaled[features_to_scale] = final_scaler.fit_transform(raw_data_train_scaled[features_to_scale])

print(f"✓ Dataset scalato creato: raw_data_train_scaled")
print(f"  Features scalate: {len(features_to_scale)}")
print(f"  Features non scalate: {len(raw_data_train.columns) - len(features_to_scale)}")

# Salva lo scaler
import pickle
with open('standard_scaler.pkl', 'wb') as f:
    pickle.dump(final_scaler, f)
print("✓ Scaler salvato in 'standard_scaler.pkl'")

### Scaling finale: strategia mista

Sulla base dell'analisi outlier ho già stabilito che per le due variabili di ritardo (con 13–14% di outlier IQR e code lunghe) lo `StandardScaler` non è ideale, perché media e deviazione standard sono fortemente influenzate dalla coda. Applico quindi una **strategia mista**:

- `RobustScaler` (basato su mediana e IQR) per **`Departure Delay`** e **`Arrival Delay`**;
- `StandardScaler` per le altre 16 feature numeriche (4 continue stabili + 12 rating ripuliti dagli `0`).

Le flag binarie `*_zeroflag` create al passo precedente **non vengono scalate** (sono già 0/1) e vengono aggiunte come colonne aggiuntive nel dataset pronto per il modelling. Il fitting è fatto solo sul training set; gli stessi scaler verranno poi applicati al test set.

In [ ]:
# ============================================================
# SCALING FINALE - STRATEGIA MISTA (Robust + Standard)
# ============================================================
print("SCALING FINALE - STRATEGIA MISTA")
print("=" * 60)

# Feature suddivise per scaler
robust_features = ['Departure Delay in Minutes', 'Arrival Delay in Minutes']
standard_features = [
    'Age', 'Flight Distance',
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness'
]

# Parto da raw_data_train_clean (con i 0->NaN->mediana e le _zeroflag gia' create)
# Imputo eventuali NaN residui (Arrival Delay in Minutes) con la mediana di train
raw_data_train_scaled = raw_data_train_clean.copy()
for col in robust_features + standard_features:
    if raw_data_train_scaled[col].isnull().sum() > 0:
        med = raw_data_train_scaled[col].median()
        raw_data_train_scaled[col] = raw_data_train_scaled[col].fillna(med)

# Fitting separato dei due scaler (solo su training)
robust_scaler  = RobustScaler()
standard_scaler = StandardScaler()

raw_data_train_scaled[robust_features]   = robust_scaler.fit_transform(raw_data_train_scaled[robust_features])
raw_data_train_scaled[standard_features] = standard_scaler.fit_transform(raw_data_train_scaled[standard_features])

# Sostituisco il vecchio final_scaler con un contenitore dei due scaler
final_scaler = {
    "robust": robust_scaler,
    "standard": standard_scaler,
    "robust_features": robust_features,
    "standard_features": standard_features,
    "zero_flag_cols": zero_flag_cols,
    "median_imputation_map": median_imputation_map,
}

# Aggiorno la lista features_to_scale per coerenza con il resto del notebook
features_to_scale = robust_features + standard_features

print(f"RobustScaler applicato a:  {robust_features}")
print(f"StandardScaler applicato a {len(standard_features)} feature numeriche.")
print(f"Zero-flag aggiunte come passthrough: {len(zero_flag_cols)} colonne.")

# Verifica numerica
check = pd.DataFrame({
    "feature": features_to_scale,
    "mean_after":   raw_data_train_scaled[features_to_scale].mean().round(4).values,
    "std_after":    raw_data_train_scaled[features_to_scale].std().round(4).values,
    "median_after": raw_data_train_scaled[features_to_scale].median().round(4).values,
})
print("\nVerifica statistiche dopo scaling (RobustScaler -> mediana 0; StandardScaler -> media 0, std 1):")
print(check.to_string(index=False))


## 2.9 Bilanciamento del Dataset

In [ ]:
# Analisi del bilanciamento del target
print("ANALISI BILANCIAMENTO TARGET")
print("=" * 60)

target_counts = raw_data_train["satisfaction"].value_counts()
target_pct = raw_data_train["satisfaction"].value_counts(normalize=True) * 100

print(f"\nDistribuzione del target:")
for label, count in target_counts.items():
    print(f"  {label}: {count} ({target_pct[label]:.1f}%)")

ratio = target_pct.max() / target_pct.min()
print(f"\nRapporto maggioranza/minoranza: {ratio:.2f}")

# Visualizzazione
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot
colors = ['#2196F3', '#FF5722']
bars = axes[0].bar(target_counts.index, target_counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title("Distribuzione del Target", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Conteggio")
for bar, count, pct in zip(bars, target_counts.values, target_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                 f'{count}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)

# Pie chart
axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=[0.03, 0.03],
            textprops={'fontsize': 11})
axes[1].set_title("Proporzione del Target", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nCONCLUSIONE:")
print("Il dataset e' moderatamente sbilanciato (~43% satisfied vs ~57% dissatisfied).")
print("Con un rapporto di ~1.3:1, non e' necessario applicare tecniche di ricampionamento")
print("(SMOTE, undersampling, ecc.). L'accuracy resta una metrica affidabile.")

Il dataset presenta uno sbilanciamento moderato tra le due classi target: circa il 43% dei passeggeri risulta `satisfied` e il 57% `neutral or dissatisfied`. Con un rapporto maggioranza/minoranza di circa 1.3:1, non si rende necessario l'uso di tecniche di ricampionamento (SMOTE, undersampling). L'accuracy può essere usata come metrica principale, affiancata da precision, recall e F1-score per completezza.

## 2.10 Correlazione tra Features

In [ ]:
# Matrice di correlazione tra features numeriche
print("ANALISI CORRELAZIONE TRA FEATURES")
print("=" * 60)

# Seleziona solo le colonne numeriche (escluso id e target encoded)
numeric_cols = features_to_scale  # 18 features (4 continue + 14 rating)

corr_matrix = raw_data_train[numeric_cols].corr()

# Heatmap
plt.figure(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            square=True, cbar_kws={"shrink": 0.8})
plt.title("Matrice di Correlazione - Features Numeriche", fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Identifica coppie con alta correlazione (|r| > 0.7)
print("\nCOPPIE CON ALTA CORRELAZIONE (|r| > 0.7):")
print("-" * 50)

high_corr_pairs = []
for i in range(len(numeric_cols)):
    for j in range(i+1, len(numeric_cols)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            high_corr_pairs.append((numeric_cols[i], numeric_cols[j], r))
            print(f"  {numeric_cols[i]} <-> {numeric_cols[j]}: r = {r:.4f}")

if not high_corr_pairs:
    print("  Nessuna coppia con |r| > 0.7")

# Correlazione di ciascuna feature con il target (encoded)
print("\n\nCORRELAZIONE FEATURES vs TARGET:")
print("-" * 50)

target_binary = (raw_data_train["satisfaction"] == "satisfied").astype(int)
corr_with_target = raw_data_train[numeric_cols].corrwith(target_binary).sort_values(ascending=False)

for feat, r in corr_with_target.items():
    bar = "+" * int(abs(r) * 40)
    sign = "+" if r > 0 else "-"
    print(f"  {feat:40s} {sign}{abs(r):.3f}  {bar}")

# Visualizzazione
plt.figure(figsize=(10, 8))
corr_with_target.sort_values().plot(kind='barh', color=['#FF5722' if v < 0 else '#2196F3' for v in corr_with_target.sort_values()])
plt.title("Correlazione di Pearson: Features vs Target (satisfied=1)", fontsize=13, fontweight='bold')
plt.xlabel("Coefficiente di correlazione")
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nCOMMENTO SULLA MULTICOLLINEARITA':")
print("Departure Delay e Arrival Delay sono fortemente correlati (r ~ 0.97).")
print("Questo e' atteso: un ritardo in partenza causa quasi sempre un ritardo all'arrivo.")
print("Per i modelli lineari, potrebbe essere utile rimuoverne uno; per i modelli ad albero,")
print("la multicollinearita' non e' un problema significativo.")

La heatmap evidenzia che le 18 features numeriche sono in gran parte poco correlate tra loro, ad eccezione della coppia Departure Delay / Arrival Delay (r ≈ 0.97), il che è logico. La correlazione con il target mostra che le features legate ai servizi a bordo (Online boarding, Inflight entertainment, Seat comfort) hanno la correlazione positiva più forte con la soddisfazione, mentre i ritardi hanno correlazione negativa debole.

## 2.11 Feature Selection

In [ ]:
from scipy.stats import ttest_ind, chi2_contingency
from sklearn.feature_selection import mutual_info_classif
import warnings
warnings.filterwarnings('ignore')

print("FEATURE SELECTION")
print("=" * 60)

# Target binario
target_binary = (raw_data_train["satisfaction"] == "satisfied").astype(int)
group_0 = target_binary == 0
group_1 = target_binary == 1

# -------------------------------------------------------
# 1. T-TEST per features numeriche
# -------------------------------------------------------
print("\n1. T-TEST (Features Numeriche vs Target Binario)")
print("-" * 50)

ttest_results = {}
for col in features_to_scale:
    vals_0 = raw_data_train.loc[group_0, col].dropna()
    vals_1 = raw_data_train.loc[group_1, col].dropna()
    t_stat, p_val = ttest_ind(vals_0, vals_1, equal_var=False)
    ttest_results[col] = {"t_stat": abs(t_stat), "p_value": p_val}

ttest_df = pd.DataFrame(ttest_results).T.sort_values("t_stat", ascending=False)
ttest_df["rank"] = range(1, len(ttest_df) + 1)
print(ttest_df[["t_stat", "p_value", "rank"]].to_string())

# -------------------------------------------------------
# 2. CHI-SQUARE + CRAMER'S V per features categoriche
# -------------------------------------------------------
print("\n\n2. CHI-SQUARE + CRAMER'S V (Features Categoriche)")
print("-" * 50)

chi2_results = {}
for col in categorical_cols:
    ct = pd.crosstab(raw_data_train[col], raw_data_train["satisfaction"])
    chi2, p_val, dof, _ = chi2_contingency(ct)
    n = ct.sum().sum()
    min_dim = min(ct.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim))
    chi2_results[col] = {"chi2": chi2, "p_value": p_val, "cramers_v": cramers_v}

chi2_df = pd.DataFrame(chi2_results).T.sort_values("cramers_v", ascending=False)
print(chi2_df.to_string())

# -------------------------------------------------------
# 3. MUTUAL INFORMATION
# -------------------------------------------------------
print("\n\n3. MUTUAL INFORMATION (Tutte le Features)")
print("-" * 50)

# Prepara dataset completo per MI: numeriche + encoded
mi_features = features_to_scale.copy()

# Aggiungi encoded categoriche
for col in binary_cols:
    col_enc = f"{col}_encoded"
    if col_enc in raw_data_encoded.columns:
        mi_features.append(col_enc)

# Aggiungi one-hot di Class
class_dummies = [c for c in raw_data_encoded.columns if c.startswith("Class_")]
mi_features.extend(class_dummies)

X_mi = raw_data_encoded[mi_features].copy()
X_mi = X_mi.fillna(X_mi.median())

mi_scores = mutual_info_classif(X_mi, target_binary, random_state=0, n_neighbors=5)
mi_df = pd.DataFrame({"feature": mi_features, "MI_score": mi_scores}).sort_values("MI_score", ascending=False)
mi_df["rank"] = range(1, len(mi_df) + 1)
print(mi_df.to_string(index=False))

# -------------------------------------------------------
# 4. CONFRONTO RANKING E VISUALIZZAZIONE
# -------------------------------------------------------
print("\n\n4. CONFRONTO RANKING")
print("-" * 50)

# Ranking dal T-test (top 10)
print("\nTop 10 features per T-test (|t-statistic|):")
for i, (feat, row) in enumerate(ttest_df.head(10).iterrows(), 1):
    print(f"  {i}. {feat} (|t| = {row['t_stat']:.2f})")

print("\nTop 10 features per Mutual Information:")
for i, row in mi_df.head(10).iterrows():
    print(f"  {int(row['rank'])}. {row['feature']} (MI = {row['MI_score']:.4f})")

# Visualizzazione comparativa
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# T-test
ttest_top = ttest_df.head(10)
axes[0].barh(range(len(ttest_top)), ttest_top["t_stat"].values, color='#2196F3')
axes[0].set_yticks(range(len(ttest_top)))
axes[0].set_yticklabels(ttest_top.index)
axes[0].set_xlabel("|t-statistic|")
axes[0].set_title("T-Test Ranking", fontweight='bold')
axes[0].invert_yaxis()

# Chi-square (Cramer's V)
axes[1].barh(range(len(chi2_df)), chi2_df["cramers_v"].values, color='#FF9800')
axes[1].set_yticks(range(len(chi2_df)))
axes[1].set_yticklabels(chi2_df.index)
axes[1].set_xlabel("Cramer's V")
axes[1].set_title("Chi-Square Ranking (Categoriche)", fontweight='bold')
axes[1].invert_yaxis()

# MI
mi_top = mi_df.head(10)
axes[2].barh(range(len(mi_top)), mi_top["MI_score"].values, color='#4CAF50')
axes[2].set_yticks(range(len(mi_top)))
axes[2].set_yticklabels(mi_top["feature"].values)
axes[2].set_xlabel("MI Score")
axes[2].set_title("Mutual Information Ranking", fontweight='bold')
axes[2].invert_yaxis()

plt.suptitle("Confronto Feature Selection: T-Test vs Chi-Square vs Mutual Information",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# -------------------------------------------------------
# -------------------------------------------------------
# 5. SELEZIONE FEATURES PER STRADA 2 (SELEZIONE IBRIDA)
# -------------------------------------------------------
# Osservazione critica: il T-test confronta solo feature NUMERICHE.
# Le top 4 dal solo T-test sarebbero: Online boarding, Inflight entertainment,
# Seat comfort, On-board service.
# Pero' dal Chi-square risulta che Class (Cramer's V = 0.505) e
# Type of Travel (V = 0.449) sono le variabili PIU' discriminanti del dataset.
# Escluderle dalla Strada 2 significa buttare via l'informazione piu' forte.
#
# Adotto allora una SELEZIONE IBRIDA: prendo le top 2 numeriche dal T-test
# (rating con piu' p-value < 0.001) + le top 2 categoriche dal Chi-square (per Cramer's V).
# Questo rende il confronto Strada 1 vs Strada 2 piu' equo e piu' informativo.

# Top 2 numeriche dal T-test
top2_numeric = list(ttest_df.head(2).index)

# Top 2 categoriche dal Chi-square (per Cramer's V): Class e Type of Travel.
# In X_train la categorica 'Type of Travel' e' codificata come 'Type of Travel_encoded',
# mentre 'Class' diventa 'Class_Business' (drop_first implicito tramite ranking).
top2_categorical_raw = list(chi2_df.head(2).index)
_cat_to_col = {
    "Class":          "Class_Business",       # one-hot principale
    "Type of Travel": "Type of Travel_encoded",
    "Customer Type":  "Customer Type_encoded",
    "Gender":         "Gender_encoded",
}
top2_categorical = [_cat_to_col[c] for c in top2_categorical_raw]

# Composizione finale: 4 feature totali per la Strada 2
top4_features = top2_numeric + top2_categorical

print(f"Top 2 numeriche (T-test):       {top2_numeric}")
print(f"Top 2 categoriche (Chi-square): {top2_categorical_raw} -> {top2_categorical}")
print(f"\nFeatures selezionate per Strada 2 (mix numeriche+categoriche):")
print(f"  {top4_features}")
print("\nLa Strada 1 usera' invece tutte le features disponibili.")


I tre metodi di feature selection danno indicazioni complementari:

- Il **T-test** (numeriche) premia i rating di servizio: **Online boarding**, **Inflight entertainment**, **Seat comfort**, **On-board service**, **Leg room service** e **Cleanliness**.
- Il **Chi-square** (categoriche) indica invece che **Class** (Cramer's V = 0.505) e **Type of Travel** (V = 0.449) sono **le variabili più discriminanti dell'intero dataset**, superiori a qualsiasi rating numerico.
- La **Mutual Information** (che cattura anche relazioni non lineari) conferma il ranking, mettendo `Online boarding`, `Inflight wifi service`, `Class` e `Type of Travel` nelle prime quattro posizioni.

Per la **Strada 2** ho quindi scelto una **selezione ibrida**: le 2 feature numeriche più forti dal T-test + le 2 categoriche più forti dal Chi-square. Limitarsi al solo T-test (come prima ipotesi) avrebbe escluso `Class` e `Type of Travel`, ovvero l'informazione più rilevante secondo gli altri due test. La nuova selezione rende il confronto Strada 1 vs Strada 2 sostanzialmente più equo.

## 2.12 Pipeline Dati Unificato

In questa sezione costruisco i dataset ML-ready per il modelling, unificando le features numeriche scalate con quelle categoriche encoded. Carico anche il test set e applico le stesse trasformazioni.

In [ ]:
# ============================================================
# PIPELINE DATI UNIFICATO - Creazione dataset ML-ready
# ============================================================
# Usa gli scaler misti definiti sopra (RobustScaler sulle delay,
# StandardScaler sulle altre numeriche) e include le flag *_zeroflag.
print("PIPELINE DATI UNIFICATO")
print("=" * 60)

# -------------------------------------------------------
# 1. CODIFICA TARGET
# -------------------------------------------------------
y_train = (raw_data_train["satisfaction"] == "satisfied").astype(int)
print(f"Target encoded: satisfied -> 1, neutral or dissatisfied -> 0")
print(f"  y_train shape: {y_train.shape}, distribuzione: {y_train.value_counts().to_dict()}")

# -------------------------------------------------------
# 2. COSTRUZIONE X_train
# -------------------------------------------------------
# a) 18 features numeriche scalate (gia' presenti in raw_data_train_scaled)
X_train_numeric = raw_data_train_scaled[features_to_scale].copy()

# b) flag binarie *_zeroflag (non vanno scalate)
X_train_zeroflags = raw_data_train_scaled[zero_flag_cols].copy()

# c) 3 features binarie encoded (Gender, Customer Type, Type of Travel)
X_train_binary = raw_data_encoded[[f"{col}_encoded" for col in binary_cols]].copy()

# d) 3 features one-hot per Class
class_dummies_train = pd.get_dummies(raw_data_train["Class"], prefix="Class").astype(int)

X_train = pd.concat(
    [X_train_numeric, X_train_zeroflags, X_train_binary, class_dummies_train],
    axis=1,
)

print(f"\nX_train composto da:")
print(f"  - {X_train_numeric.shape[1]} features numeriche scalate (RobustScaler/StandardScaler)")
print(f"  - {X_train_zeroflags.shape[1]} flag binarie *_zeroflag")
print(f"  - {X_train_binary.shape[1]} features binarie encoded")
print(f"  - {class_dummies_train.shape[1]} features one-hot (Class)")
print(f"  => Totale: {X_train.shape[1]} features, {X_train.shape[0]} campioni")

# -------------------------------------------------------
# 3. CARICAMENTO E PREPARAZIONE TEST SET
# -------------------------------------------------------
print("\n" + "-" * 60)
print("PREPARAZIONE TEST SET")
print("-" * 60)

raw_data_test = pd.read_csv("test.csv")
raw_data_test = raw_data_test.drop(columns=["Unnamed: 0"], errors='ignore')

y_test = (raw_data_test["satisfaction"] == "satisfied").astype(int)
print(f"Test set caricato: {raw_data_test.shape[0]} campioni")
print(f"  y_test distribuzione: {y_test.value_counts().to_dict()}")

# a) Applico la stessa pulizia 'rating == 0 -> NaN -> mediana train' fatta sul training
raw_data_test_clean = raw_data_test.copy()
for c, median_val in median_imputation_map.items():
    raw_data_test_clean[f"{c}_zeroflag"] = (raw_data_test_clean[c] == 0).astype(int)
    raw_data_test_clean[c] = raw_data_test_clean[c].replace(0, np.nan)
    raw_data_test_clean[c] = raw_data_test_clean[c].fillna(median_val)

# b) Imputo NaN residui (es. Arrival Delay) con la mediana del TRAIN set
for col in features_to_scale:
    if raw_data_test_clean[col].isnull().sum() > 0:
        train_median = raw_data_train[col].median()
        n_imp = int(raw_data_test_clean[col].isnull().sum())
        raw_data_test_clean[col] = raw_data_test_clean[col].fillna(train_median)
        print(f"  Imputato {col}: {n_imp} NaN con mediana train ({train_median:.2f})")

# c) Applico i due scaler (fittati solo sul train)
X_test_numeric = raw_data_test_clean[features_to_scale].copy()
X_test_numeric[robust_features]   = robust_scaler.transform(X_test_numeric[robust_features])
X_test_numeric[standard_features] = standard_scaler.transform(X_test_numeric[standard_features])

# d) Zero-flag del test
X_test_zeroflags = raw_data_test_clean[zero_flag_cols].copy()

# e) Encode binarie con gli stessi encoder del train
X_test_binary_dict = {}
for col in binary_cols:
    X_test_binary_dict[f"{col}_encoded"] = le_dict[col].transform(raw_data_test[col])
X_test_binary = pd.DataFrame(X_test_binary_dict, index=raw_data_test.index)

# f) One-hot per Class (stesse colonne del train)
class_dummies_test = pd.get_dummies(raw_data_test["Class"], prefix="Class").astype(int)
for col in class_dummies_train.columns:
    if col not in class_dummies_test.columns:
        class_dummies_test[col] = 0
class_dummies_test = class_dummies_test[class_dummies_train.columns]

X_test = pd.concat(
    [X_test_numeric, X_test_zeroflags, X_test_binary, class_dummies_test],
    axis=1,
)
# Allineo colonne train/test per sicurezza
X_test = X_test[X_train.columns]
print(f"\nX_test shape: {X_test.shape}")

# -------------------------------------------------------
# 4. DATASET STRADA 2 (selezione ibrida - definita piu' avanti)
# -------------------------------------------------------
# La lista top4_features viene aggiornata nella sezione di feature selection.
X_train_strada2 = X_train[top4_features].copy()
X_test_strada2  = X_test[top4_features].copy()
print(f"\nStrada 2 features ({len(top4_features)}): {top4_features}")
print(f"  X_train_strada2: {X_train_strada2.shape}")
print(f"  X_test_strada2:  {X_test_strada2.shape}")

# -------------------------------------------------------
# 5. VERIFICA FINALE
# -------------------------------------------------------
print("\n" + "=" * 60)
print("VERIFICA DATASET ML-READY")
print("=" * 60)
print(f"  X_train:         {X_train.shape}  |  y_train: {y_train.shape}")
print(f"  X_test:          {X_test.shape}  |  y_test:  {y_test.shape}")
print(f"  X_train_strada2: {X_train_strada2.shape}")
print(f"  X_test_strada2:  {X_test_strada2.shape}")
print(f"  NaN in X_train:  {X_train.isnull().sum().sum()}")
print(f"  NaN in X_test:   {X_test.isnull().sum().sum()}")
print(f"\nFeatures X_train: {list(X_train.columns)}")


# 3.0 Scelta della Metrica di Valutazione

Per valutare i modelli di classificazione utilizzo le seguenti metriche:

- **Accuracy**: percentuale di predizioni corrette sul totale. È la metrica principale perché il dataset è sufficientemente bilanciato (~43/57).
- **Precision**: tra le predizioni positive, quante sono effettivamente positive. Importante per minimizzare i falsi positivi.
- **Recall (Sensitivity)**: tra i campioni effettivamente positivi, quanti sono stati correttamente identificati. Importante per minimizzare i falsi negativi.
- **F1-Score**: media armonica di Precision e Recall. Utile quando si cerca un equilibrio tra le due.
- **AUC-ROC**: area sotto la curva ROC (Receiver Operating Characteristic). Misura la capacità discriminativa del modello indipendentemente dalla soglia di classificazione.

**Perché l'Accuracy come metrica principale?**
Con uno sbilanciamento moderato (rapporto ~1.3:1), l'accuracy non è distorta come lo sarebbe in dataset fortemente sbilanciati (es. 95/5). Riporto comunque sempre anche Precision, Recall, F1 e AUC-ROC per una valutazione completa.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier

print("Librerie per modellazione e metriche importate con successo.")
print(f"\nModelli che verranno confrontati:")
print("  1. Logistic Regression")
print("  2. Random Forest")
print("  3. AdaBoost")
print("  4. Gradient Boosting")
print(f"\nMetriche: Accuracy (principale), Precision, Recall, F1-Score, AUC-ROC")

# 4.0 Modelli, Training e Ottimizzazione

## 4.1 Strada 1: Tutte le Features

In questa strada utilizzo tutte le **37 features** disponibili: 18 numeriche scalate (di cui 2 con RobustScaler e 16 con StandardScaler), 13 flag binarie `*_zeroflag` per i rating non applicabili, 3 binary encoded (Gender, Customer Type, Type of Travel) e 3 one-hot per Class.

In [ ]:
# ============================================================
# STRADA 1 - SPOT CHECK: 5-Fold StratifiedKFold
# ============================================================
print("STRADA 1: SPOT CHECK - 5-Fold Cross-Validation")
print("=" * 60)

models_dict = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=SEED),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=SEED, algorithm='SAMME'),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=SEED)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_results_s1 = {}
for name, model in models_dict.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results_s1[name] = scores
    print(f"\n{name}:")
    print(f"  Accuracy per fold: {[f'{s:.4f}' for s in scores]}")
    print(f"  Media: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Boxplot comparativo
fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot([cv_results_s1[name] for name in models_dict.keys()],
                labels=models_dict.keys(), patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='navy'),
                medianprops=dict(color='red', linewidth=2))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_title("Strada 1: Spot Check - 5-Fold CV Accuracy", fontsize=13, fontweight='bold')
ax.set_ylabel("Accuracy")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Selezione top 2
mean_scores_s1 = {name: scores.mean() for name, scores in cv_results_s1.items()}
sorted_models_s1 = sorted(mean_scores_s1.items(), key=lambda x: x[1], reverse=True)

print("\nRANKING STRADA 1:")
for i, (name, score) in enumerate(sorted_models_s1, 1):
    marker = " <-- TOP 2" if i <= 2 else ""
    print(f"  {i}. {name}: {score:.4f}{marker}")

top2_names_s1 = [sorted_models_s1[0][0], sorted_models_s1[1][0]]
print(f"\nTop 2 selezionati per GridSearch: {top2_names_s1}")

In [ ]:
# ============================================================
# STRADA 1 - GRIDSEARCH SUI TOP 2
# ============================================================
print("STRADA 1: GRIDSEARCH SUI TOP 2 MODELLI")
print("=" * 60)

# Definizione griglie iperparametri
param_grids = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["lbfgs", "liblinear"]
    },
    "Random Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [10, 20, None],
        "min_samples_split": [2, 5]
    },
    "AdaBoost": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.1, 0.5, 1.0]
    },
    "Gradient Boosting": {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.1, 0.2]
    }
}

best_models_s1 = {}
for name in top2_names_s1:
    print(f"\n{'='*50}")
    print(f"GridSearchCV: {name}")
    print(f"{'='*50}")

    base_model = models_dict[name]
    grid = param_grids[name]

    gs = GridSearchCV(base_model, grid, cv=cv, scoring='accuracy',
                      n_jobs=-1, verbose=0, refit=True)
    gs.fit(X_train, y_train)

    best_models_s1[name] = gs.best_estimator_
    print(f"  Best params: {gs.best_params_}")
    print(f"  Best CV accuracy: {gs.best_score_:.4f}")

print("\nGridSearch Strada 1 completato.")

In [ ]:
# ============================================================
# STRADA 1 - VALUTAZIONE SUL TEST SET
# ============================================================
print("STRADA 1: VALUTAZIONE SUL TEST SET")
print("=" * 60)

results_s1 = {}

for name, model in best_models_s1.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results_s1[name] = {
        "Accuracy": acc, "Precision": prec, "Recall": rec,
        "F1-Score": f1, "AUC-ROC": auc,
        "y_pred": y_pred, "y_proba": y_proba
    }

    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["dissatisfied", "satisfied"]))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["dissatisfied", "satisfied"],
                yticklabels=["dissatisfied", "satisfied"])
    plt.title(f"Confusion Matrix - {name} (Strada 1)", fontweight='bold')
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

## 4.2 Strada 2: 4 Features dalla Selezione Ibrida

In questa strada utilizzo le **4 features più discriminanti** ottenute incrociando T-test (per le numeriche) e Chi-square (per le categoriche), per valutare se un modello più parsimonioso ma teoricamente migliore della sola top-4 T-test possa raggiungere performance comparabili alla Strada 1.


In [ ]:
# ============================================================
# STRADA 2 - SPOT CHECK: 5-Fold StratifiedKFold
# ============================================================
print("STRADA 2: SPOT CHECK - 5-Fold Cross-Validation (Selezione Ibrida)")
print("=" * 60)
print(f"Features utilizzate: {top4_features}")

models_dict_s2 = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=SEED),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=SEED, algorithm='SAMME'),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=SEED)
}

cv_results_s2 = {}
for name, model in models_dict_s2.items():
    scores = cross_val_score(model, X_train_strada2, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results_s2[name] = scores
    print(f"\n{name}:")
    print(f"  Accuracy per fold: {[f'{s:.4f}' for s in scores]}")
    print(f"  Media: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Boxplot comparativo
fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot([cv_results_s2[name] for name in models_dict_s2.keys()],
                labels=models_dict_s2.keys(), patch_artist=True,
                medianprops=dict(color='red', linewidth=2))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_title("Strada 2: Spot Check - 5-Fold CV Accuracy (Selezione Ibrida)", fontsize=13, fontweight='bold')
ax.set_ylabel("Accuracy")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Selezione top 2
mean_scores_s2 = {name: scores.mean() for name, scores in cv_results_s2.items()}
sorted_models_s2 = sorted(mean_scores_s2.items(), key=lambda x: x[1], reverse=True)

print("\nRANKING STRADA 2:")
for i, (name, score) in enumerate(sorted_models_s2, 1):
    marker = " <-- TOP 2" if i <= 2 else ""
    print(f"  {i}. {name}: {score:.4f}{marker}")

top2_names_s2 = [sorted_models_s2[0][0], sorted_models_s2[1][0]]
print(f"\nTop 2 selezionati per GridSearch: {top2_names_s2}")

In [ ]:
# ============================================================
# STRADA 2 - GRIDSEARCH SUI TOP 2
# ============================================================
print("STRADA 2: GRIDSEARCH SUI TOP 2 MODELLI")
print("=" * 60)

best_models_s2 = {}
for name in top2_names_s2:
    print(f"\n{'='*50}")
    print(f"GridSearchCV: {name}")
    print(f"{'='*50}")

    base_model = models_dict_s2[name]
    grid = param_grids[name]

    gs = GridSearchCV(base_model, grid, cv=cv, scoring='accuracy',
                      n_jobs=-1, verbose=0, refit=True)
    gs.fit(X_train_strada2, y_train)

    best_models_s2[name] = gs.best_estimator_
    print(f"  Best params: {gs.best_params_}")
    print(f"  Best CV accuracy: {gs.best_score_:.4f}")

print("\nGridSearch Strada 2 completato.")

In [ ]:
# ============================================================
# STRADA 2 - VALUTAZIONE SUL TEST SET
# ============================================================
print("STRADA 2: VALUTAZIONE SUL TEST SET (Selezione Ibrida)")
print("=" * 60)

results_s2 = {}

for name, model in best_models_s2.items():
    y_pred = model.predict(X_test_strada2)
    y_proba = model.predict_proba(X_test_strada2)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results_s2[name] = {
        "Accuracy": acc, "Precision": prec, "Recall": rec,
        "F1-Score": f1, "AUC-ROC": auc,
        "y_pred": y_pred, "y_proba": y_proba
    }

    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["dissatisfied", "satisfied"]))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
                xticklabels=["dissatisfied", "satisfied"],
                yticklabels=["dissatisfied", "satisfied"])
    plt.title(f"Confusion Matrix - {name} (Strada 2)", fontweight='bold')
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

# 5.0 Valutazione Finale e Confronto

In questa sezione confronto tutti i modelli delle due strade per identificare il miglior approccio complessivo.

In [ ]:
# ============================================================
# VALUTAZIONE FINALE - CONFRONTO TUTTI I MODELLI
# ============================================================
print("VALUTAZIONE FINALE - CONFRONTO TUTTI I MODELLI")
print("=" * 60)

# Tabella riassuntiva
all_results = {}
for name, res in results_s1.items():
    all_results[f"S1 - {name}"] = {k: v for k, v in res.items() if k not in ["y_pred", "y_proba"]}
for name, res in results_s2.items():
    all_results[f"S2 - {name}"] = {k: v for k, v in res.items() if k not in ["y_pred", "y_proba"]}

comparison_df = pd.DataFrame(all_results).T.sort_values("Accuracy", ascending=False)
print("\nTABELLA CONFRONTO:")
print(comparison_df.round(4).to_string())

# -------------------------------------------------------
# BARPLOT: Accuracy + AUC-ROC
# -------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy
comparison_df["Accuracy"].sort_values().plot(kind='barh', ax=axes[0], color='#2196F3', edgecolor='black')
axes[0].set_title("Accuracy - Tutti i Modelli", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Accuracy")
for i, v in enumerate(comparison_df["Accuracy"].sort_values()):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center')

# AUC-ROC
comparison_df["AUC-ROC"].sort_values().plot(kind='barh', ax=axes[1], color='#FF9800', edgecolor='black')
axes[1].set_title("AUC-ROC - Tutti i Modelli", fontsize=13, fontweight='bold')
axes[1].set_xlabel("AUC-ROC")
for i, v in enumerate(comparison_df["AUC-ROC"].sort_values()):
    axes[1].text(v + 0.002, i, f'{v:.4f}', va='center')

plt.tight_layout()
plt.show()

# -------------------------------------------------------
# ROC CURVES - Tutti i modelli sovrapposti
# -------------------------------------------------------
plt.figure(figsize=(10, 8))

colors_roc = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0', '#FF9800', '#00BCD4', '#795548', '#607D8B']
color_idx = 0

for label, res in {**{f"S1-{k}": v for k, v in results_s1.items()},
                   **{f"S2-{k}": v for k, v in results_s2.items()}}.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_proba"])
    auc_val = res["AUC-ROC"]
    plt.plot(fpr, tpr, label=f'{label} (AUC={auc_val:.4f})',
             color=colors_roc[color_idx % len(colors_roc)], linewidth=2)
    color_idx += 1

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.5)')
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves - Confronto Tutti i Modelli", fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# -------------------------------------------------------
# DISCUSSIONE FINALE
# -------------------------------------------------------
best_model_name = comparison_df["Accuracy"].idxmax()
best_accuracy = comparison_df.loc[best_model_name, "Accuracy"]
best_auc = comparison_df.loc[best_model_name, "AUC-ROC"]

print("\n" + "=" * 60)
print("DISCUSSIONE FINALE")
print("=" * 60)

print(f"\nMiglior modello complessivo: {best_model_name}")
print(f"  Accuracy: {best_accuracy:.4f}")
print(f"  AUC-ROC:  {best_auc:.4f}")

# Confronto strade
s1_accs = [v["Accuracy"] for v in results_s1.values()]
s2_accs = [v["Accuracy"] for v in results_s2.values()]

print(f"\nAccuracy media Strada 1 (tutte le features): {np.mean(s1_accs):.4f}")
print(f"Accuracy media Strada 2 (top 4 features):    {np.mean(s2_accs):.4f}")

if np.mean(s1_accs) > np.mean(s2_accs):
    print("\nLa Strada 1 (tutte le features) ottiene performance superiori, come atteso.")
    print("Le features aggiuntive forniscono informazione complementare utile ai modelli.")
else:
    print("\nLa Strada 2 (top 4 features) ottiene performance comparabili o superiori.")
    print("Cio' suggerisce che i 4 driver principali catturano la maggior parte dell'informazione.")

print("\nRACCOMANDAZIONE:")
print(f"Si raccomanda il modello {best_model_name} per il deployment,")
print(f"in quanto offre il miglior bilanciamento tra accuracy ({best_accuracy:.4f})")
print(f"e capacita' discriminativa (AUC-ROC = {best_auc:.4f}).")
print("\nLa Strada 2, pur con performance leggermente inferiori, offre il vantaggio")
print("della semplicita' e interpretabilita', richiedendo solo 4 features.")

## 5.1 Feature importance del modello vincitore

Il miglior modello è risultato il **Gradient Boosting della Strada 1**. Estraggo ora la sua `feature_importances_` per due motivi: (1) confermare che il modello "vede" come driver le stesse variabili che l'EDA e i test statistici (T-test, Chi-square, Mutual Information) avevano già indicato come più rilevanti; (2) avere una base interpretativa per le raccomandazioni di business della sezione finale.

In [ ]:
# ============================================================
# FEATURE IMPORTANCE DEL MODELLO VINCITORE (S1 - Gradient Boosting)
# ============================================================
best_overall = best_models_s1["Gradient Boosting"]

fi = pd.DataFrame({
    "feature":    X_train.columns,
    "importance": best_overall.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("FEATURE IMPORTANCE - GRADIENT BOOSTING (Strada 1)")
print("=" * 60)
print(fi.to_string(index=False))

# Grafico
plt.figure(figsize=(10, 8))
fi_sorted = fi.sort_values("importance")
colors_fi = ['#4CAF50' if v >= fi['importance'].median() else '#BDBDBD'
             for v in fi_sorted['importance']]
plt.barh(fi_sorted['feature'], fi_sorted['importance'], color=colors_fi, edgecolor='black')
plt.title("Feature Importance - Gradient Boosting (Strada 1)", fontsize=13, fontweight='bold')
plt.xlabel("Importance (Gini-based)")
plt.tight_layout()
plt.show()

# Confronto qualitativo con T-test e Chi-square
print("\nConfronto con altri ranking di rilevanza:")
print("-" * 60)
top5_fi = list(fi.head(5)["feature"])
top5_tt = list(ttest_df.head(5).index)
top5_cs = list(chi2_df.head(5).index)
print(f"  Top 5 Gradient Boosting:  {top5_fi}")
print(f"  Top 5 T-test (numeriche): {top5_tt}")
print(f"  Top categoriche Chi-sq:   {top5_cs}")
print("\nIl modello finale conferma in larga parte cio' che era emerso dall'analisi")
print("statistica preliminare: i rating di servizio (Online boarding, Inflight entertainment,")
print("Seat comfort) e le variabili contestuali (Class, Type of Travel) sono i veri driver.")


## 5.2 Modello operativo "pre-volo" - controllo di leakage

Un punto critico emerso durante l'EDA: i rating di servizio (Wi-Fi, Seat comfort, Online boarding, Cleanliness, ecc.) sono **valutazioni post-volo** raccolte sulla stessa survey in cui il passeggero dichiara la propria soddisfazione. Usarli come predittori di `satisfaction` produce un modello *diagnostico* molto accurato (quello a 96.4%) ma poco utile in scenari **pre-volo**, dove la compagnia non conosce ancora i rating.

Per stimare quanto del segnale dipende dalle sole variabili note **prima** del volo, addestro un secondo modello usando esclusivamente: `Age`, `Customer Type`, `Type of Travel`, `Class`, `Flight Distance`, `Gender`, e i ritardi (che possono essere stimati a partire dallo schedule). Il delta di accuracy tra il modello "completo" e quello "pre-volo" misura quanto pesa l'informazione raccolta a bordo.

In [ ]:
# ============================================================
# MODELLO OPERATIVO PRE-VOLO
# Solo feature note PRIMA dell'erogazione del servizio.
# ============================================================
prevolo_features = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
    "Gender_encoded",
    "Customer Type_encoded",
    "Type of Travel_encoded",
    "Class_Business", "Class_Eco", "Class_Eco Plus",
]

X_train_prevolo = X_train[prevolo_features].copy()
X_test_prevolo  = X_test[prevolo_features].copy()

print("MODELLO OPERATIVO PRE-VOLO")
print("=" * 60)
print(f"Feature usate ({len(prevolo_features)}): {prevolo_features}")

# Stesso pool di modelli, stessa cross-validation
cv_results_pre = {}
for name, model in models_dict.items():
    scores = cross_val_score(model, X_train_prevolo, y_train, cv=cv,
                             scoring='accuracy', n_jobs=-1)
    cv_results_pre[name] = scores
    print(f"  {name:22s} CV accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")

# Tuning del miglior modello pre-volo
best_name_pre = max(cv_results_pre, key=lambda n: cv_results_pre[n].mean())
print(f"\nMiglior modello pre-volo (spot-check): {best_name_pre}")

gs_pre = GridSearchCV(models_dict[best_name_pre], param_grids[best_name_pre],
                      cv=cv, scoring='accuracy', n_jobs=-1, refit=True)
gs_pre.fit(X_train_prevolo, y_train)
model_pre = gs_pre.best_estimator_
print(f"  Best params: {gs_pre.best_params_}")
print(f"  Best CV accuracy: {gs_pre.best_score_:.4f}")

# Valutazione sul test set
y_pred_pre = model_pre.predict(X_test_prevolo)
y_proba_pre = model_pre.predict_proba(X_test_prevolo)[:, 1]

acc_pre  = accuracy_score(y_test, y_pred_pre)
prec_pre = precision_score(y_test, y_pred_pre)
rec_pre  = recall_score(y_test, y_pred_pre)
f1_pre   = f1_score(y_test, y_pred_pre)
auc_pre  = roc_auc_score(y_test, y_proba_pre)

print("\nValutazione sul test set:")
print(f"  Accuracy:  {acc_pre:.4f}")
print(f"  Precision: {prec_pre:.4f}")
print(f"  Recall:    {rec_pre:.4f}")
print(f"  F1-Score:  {f1_pre:.4f}")
print(f"  AUC-ROC:   {auc_pre:.4f}")

# Confronto con il modello completo
acc_full = results_s1["Gradient Boosting"]["Accuracy"]
delta = acc_full - acc_pre
print(f"\nConfronto modello COMPLETO (con rating) vs PRE-VOLO:")
print(f"  Accuracy completo : {acc_full:.4f}")
print(f"  Accuracy pre-volo : {acc_pre:.4f}")
print(f"  Delta             : {delta:+.4f}")
print("\nLetture possibili:")
print("  - Il delta misura quanto valore aggiunge l'esperienza in-flight per la predizione.")
print("  - Il modello pre-volo da' la performance realisticamente ottenibile prima del volo.")
print("  - Il modello completo resta utile come SCORING DIAGNOSTICO sulle survey post-volo.")


# 6.0 Conclusione

Il progetto ha seguito un percorso completo di data science: dall'analisi esplorativa, alla preparazione dei dati, alla selezione delle features, fino alla modellazione e ottimizzazione, con due strade parallele e un controllo esplicito del rischio di leakage.

### Risultati tecnici principali

- La **Strada 1** (tutte le features) raggiunge le performance migliori: **Gradient Boosting tunato** con Accuracy = **0.9637** e AUC-ROC = **0.9954** sul test set.
- La **Strada 2** (selezione ibrida T-test + Chi-square: `Online boarding`, `Inflight entertainment`, `Class`, `Type of Travel`) raggiunge Accuracy = **0.866** e AUC-ROC = **0.936** con sole 4 feature, dimostrando che questi quattro driver da soli catturano la maggior parte del segnale.
- Il **modello operativo pre-volo** (senza i rating raccolti a bordo) si ferma a Accuracy = **0.8028** e AUC-ROC = **0.8667**, con un delta di **-16 punti percentuali** rispetto al modello completo. È la stima più onesta di ciò che è predicibile *prima* che il passeggero metta piede sull'aereo.
- La gestione dei rating `== 0` come "non applicabile" si è rivelata corretta: la feature `Inflight wifi service_zeroflag` è nella **top 5** della feature importance del Gradient Boosting, segno che il modello distingue davvero quella categoria.
- I modelli ensemble (Random Forest, Gradient Boosting) battono sistematicamente la Logistic Regression, coerentemente con le non-linearità osservate nell'EDA (effetti soglia 3->4 in quasi tutti i rating).
- Il dataset, moderatamente sbilanciato (43/57), non ha richiesto resampling; l'accuracy rimane una metrica affidabile, affiancata da Precision, Recall, F1 e AUC-ROC.

### Raccomandazioni operative per la compagnia

Dalla feature importance del modello vincitore e dai pattern visti nell'EDA emergono tre leve concrete su cui agire:

1. **Investire sull'esperienza digitale di Online Boarding.** È la feature numero 1 in tutti e tre i ranking (T-test, Mutual Information, feature importance del GB con 41.6% di importanza). Nei dati, passare da un rating <=3 a >=4 ribalta la probabilità di essere `satisfied` (dal ~13% al ~62%): un'app stabile, QR offline, sincronizzazione PNR e gestione delle eccezioni sono le aree con il payoff più alto.
2. **Riallineare il prodotto Eco.** La probabilità di soddisfazione in Eco è ~19%, contro ~69% in Business: il divario non riguarda solo il sedile, ma il mix complessivo di servizio percepito (intrattenimento, pulizia, food&drink). Definire uno standard minimo anche in Eco è la leva con il volume di passeggeri impattati più alto.
3. **Comunicare i ritardi con anticipo, non eliminare l'evento.** I delay >30' sono il punto di rottura più frequente, ma la relazione con la soddisfazione è a soglia più che lineare. Un ritardo annunciato e gestito (rebooking, voucher, comunicazione via app) riduce il "dolore percepito" anche quando la durata oggettiva resta alta.

### Coerenza con il brief: turismo e impatto ambientale

La consegna parte dall'idea di una compagnia attenta alla **qualità del servizio e all'impatto ambientale**. I dati offrono tre agganci pertinenti:

- I voli **molto corti** (sotto i ~800 km) sono allo stesso tempo i meno apprezzati (coda bassa di `satisfied`) e i meno efficienti per passeggero-km (decollo e atterraggio pesano in proporzione molto di più sul consumo). Ridurre frequenze su queste tratte, incentivare il treno o consolidare le frequenze libera capacità su rotte più lunghe e migliora contemporaneamente CSAT e impatto ambientale.
- I **ritardi importanti** generano tempo di rullaggio, holding e ricarburazione supplementare: ogni minuto di ritardo non è solo una frizione di customer experience, ma anche emissioni e costo carburante extra. Investire sulla riduzione dei delay >30' ha quindi un ritorno doppio (soddisfazione + sostenibilità).
- L'**online boarding ben funzionante** riduce stampe cartacee, navette, code: anche qui un piccolo guadagno operativo si traduce in efficienza energetica del processo di gate.

### Limiti dichiarati

- Il modello completo ottiene un'accuracy molto alta in parte perché utilizza rating raccolti **dopo** il volo, sulla stessa survey della target: è uno scoring diagnostico, non un predittore operativo strict-sensu. Il modello pre-volo addestrato qui sopra (Accuracy 0.8028) rappresenta una baseline più realistica per un uso "in produzione".
- Le variabili `Departure Delay` e `Arrival Delay` sono fortemente collineari (r ~ 0.97). Le ho mantenute entrambe perché i modelli vincenti sono tree-based; in un contesto con modelli lineari sarebbe opportuno tenerne solo una o costruire una variabile sintetica.
- I valori `0` di alcuni rating sono stati trattati come "non applicabile" (flag binaria + imputazione mediana 1-5): l'interpretazione è supportata dai dati (e dalla feature importance), ma non da una documentazione esplicita del data dictionary.


# 7.0 Persistenza del modello finale

Salvo il miglior modello (Gradient Boosting della Strada 1) insieme ai preprocessori necessari per applicare la stessa pipeline su nuovi dati: gli scaler, gli encoder, la mappa di imputazione e la lista delle colonne attese. In questo modo il modello è auto-contenuto e riproducibile.

In [ ]:
# ============================================================
# PERSISTENZA DEL MODELLO + PIPELINE DI PREPROCESSING
# ============================================================
import joblib
import os

artifact = {
    "model":                 best_models_s1["Gradient Boosting"],
    "feature_columns":       list(X_train.columns),
    "robust_scaler":         robust_scaler,
    "standard_scaler":       standard_scaler,
    "robust_features":       robust_features,
    "standard_features":     standard_features,
    "zero_flag_cols":        zero_flag_cols,
    "median_imputation_map": median_imputation_map,
    "label_encoders":        le_dict,
    "class_dummy_columns":   list(class_dummies_train.columns),
}

out_path = "airline_satisfaction_model.joblib"
joblib.dump(artifact, out_path)
print(f"Artifact salvato in: {out_path}")
print(f"  Dimensione file: {os.path.getsize(out_path)/1e6:.2f} MB")

# Esempio di inferenza: ricarico l'artifact e predico le prime 5 righe del test set
print("\nEsempio di inferenza con l'artifact ricaricato:")
loaded = joblib.load(out_path)
sample = X_test.head(5)[loaded["feature_columns"]]
preds = loaded["model"].predict(sample)
probas = loaded["model"].predict_proba(sample)[:, 1]

demo = pd.DataFrame({
    "y_true":      y_test.head(5).values,
    "y_pred":      preds,
    "P(satisfied)": probas.round(3),
})
print(demo.to_string(index=False))
